In [1]:
# ============================================================
# LUMORA — PRE-DEPLOYMENT MODEL TESTING
# Step 1: Project Setup & Saved Model Loading
# ============================================================

from pathlib import Path
import joblib
import pandas as pd
import sklearn

# Project paths
PROJECT_ROOT = Path.home() / "Desktop" / "Lumora"

MODEL_PATH = PROJECT_ROOT / "models" / "lumora_word_char_v1.joblib"

# Load the saved model artifact
model = joblib.load(MODEL_PATH)

# Basic project information
print("PROJECT:", PROJECT_ROOT)
print("MODEL:", MODEL_PATH.name)
print("MODEL EXISTS:", MODEL_PATH.exists())
print("MODEL TYPE:", type(model).__name__)
print("SKLEARN VERSION:", sklearn.__version__)

# Model classes
print("\nSUPPORTED CATEGORIES:")
print(model.classes_)

# Verify pipeline components
print("\nPIPELINE COMPONENTS:")
for name, component in model.steps:
    print(f"- {name}: {type(component).__name__}")

PROJECT: /Users/raghad/Desktop/Lumora
MODEL: lumora_word_char_v1.joblib
MODEL EXISTS: True
MODEL TYPE: Pipeline
SKLEARN VERSION: 1.9.1

SUPPORTED CATEGORIES:
['ACCOUNT' 'CANCEL' 'CONTACT' 'DELIVERY' 'FEEDBACK' 'INVOICE' 'ORDER'
 'PAYMENT' 'REFUND' 'SHIPPING' 'SUBSCRIPTION']

PIPELINE COMPONENTS:
- features: FeatureUnion
- classifier: LogisticRegression


In [2]:
# ============================================================
# LUMORA — PRE-DEPLOYMENT MODEL TESTING
# Step 2: Inspect Existing Datasets
# ============================================================

from pathlib import Path
import pandas as pd

# Project paths
PROJECT_ROOT = Path.home() / "Desktop" / "Lumora"

DATA_PATHS = {
    "training": PROJECT_ROOT / "data" / "raw" / "customer_support_tickets.csv",
    "challenge": PROJECT_ROOT / "data" / "benchmark" / "challenge.csv",
    "development": PROJECT_ROOT / "data" / "benchmark" / "development.csv",
    "locked_test": PROJECT_ROOT / "data" / "benchmark" / "locked_test.csv",
    "development_v2": PROJECT_ROOT / "data" / "benchmark" / "development_v2.csv",
}

# Load datasets
datasets = {}

for name, path in DATA_PATHS.items():

    print(f"\n{'=' * 60}")
    print(f"DATASET: {name.upper()}")
    print(f"PATH: {path}")

    if not path.exists():
        print("STATUS: FILE NOT FOUND")
        continue

    df = pd.read_csv(path)
    datasets[name] = df

    print("STATUS: LOADED")
    print("SHAPE:", df.shape)
    print("COLUMNS:", df.columns.tolist())

    # Show category distribution when available
    category_column = (
        "category"
        if "category" in df.columns
        else "expected_category"
        if "expected_category" in df.columns
        else None
    )

    if category_column:
        print("\nCATEGORY DISTRIBUTION:")
        print(df[category_column].value_counts().sort_index())

    # Show test type distribution when available
    if "test_type" in df.columns:
        print("\nTEST TYPE DISTRIBUTION:")
        print(df["test_type"].value_counts())

print("\n" + "=" * 60)
print("DATASET INSPECTION COMPLETED")
print("LOADED DATASETS:", list(datasets.keys()))


DATASET: TRAINING
PATH: /Users/raghad/Desktop/Lumora/data/raw/customer_support_tickets.csv
STATUS: LOADED
SHAPE: (26872, 5)
COLUMNS: ['flags', 'instruction', 'category', 'intent', 'response']

CATEGORY DISTRIBUTION:
category
ACCOUNT         5986
CANCEL           950
CONTACT         1999
DELIVERY        1994
FEEDBACK        1997
INVOICE         1999
ORDER           3988
PAYMENT         1998
REFUND          2992
SHIPPING        1970
SUBSCRIPTION     999
Name: count, dtype: int64

DATASET: CHALLENGE
PATH: /Users/raghad/Desktop/Lumora/data/benchmark/challenge.csv
STATUS: LOADED
SHAPE: (22, 11)
COLUMNS: ['example_id', 'text', 'expected_category', 'expected_intent', 'test_type', 'challenge_type', 'expected_categories', 'expected_intents', 'requires_clarification', 'annotation_rationale', 'guideline_version']

CATEGORY DISTRIBUTION:
Series([], Name: count, dtype: int64)

TEST TYPE DISTRIBUTION:
test_type
natural    22
Name: count, dtype: int64

DATASET: DEVELOPMENT
PATH: /Users/raghad/Deskto

In [6]:
# ============================================================
# LUMORA — PRE-DEPLOYMENT MODEL TESTING
# Step 3: Inspect Benchmark Schemas
# ============================================================

for name, df in datasets.items():

    print(f"\n{'=' * 55}")
    print(f"DATASET: {name.upper()}")
    print(f"ROWS: {len(df)}")

    print("\nCOLUMNS AND DATA TYPES:")

    for column, dtype in df.dtypes.items():
        print(f"- {column}: {dtype}")

    # Identify possible text columns
    text_columns = [
        col for col in df.columns
        if any(word in col.lower()
               for word in ["text", "instruction", "message", "request"])
    ]

    print("\nPOSSIBLE TEXT COLUMNS:", text_columns)

print("\nSCHEMA INSPECTION COMPLETED")


DATASET: TRAINING
ROWS: 26872

COLUMNS AND DATA TYPES:
- flags: str
- instruction: str
- category: str
- intent: str
- response: str

POSSIBLE TEXT COLUMNS: ['instruction']

DATASET: CHALLENGE
ROWS: 22

COLUMNS AND DATA TYPES:
- example_id: str
- text: str
- expected_category: float64
- expected_intent: float64
- test_type: str
- challenge_type: str
- expected_categories: str
- expected_intents: str
- requires_clarification: bool
- annotation_rationale: str
- guideline_version: float64

POSSIBLE TEXT COLUMNS: ['text']

DATASET: DEVELOPMENT
ROWS: 27

COLUMNS AND DATA TYPES:
- example_id: str
- text: str
- expected_category: str
- expected_intent: str
- test_type: str
- challenge_type: float64
- expected_categories: float64
- expected_intents: float64
- requires_clarification: bool
- annotation_rationale: str
- guideline_version: float64

POSSIBLE TEXT COLUMNS: ['text']

DATASET: LOCKED_TEST
ROWS: 108

COLUMNS AND DATA TYPES:
- example_id: str
- text: str
- expected_category: str
- expe

In [7]:
# ============================================================
# LUMORA — PRE-DEPLOYMENT MODEL TESTING
# Step 4: Benchmark Data Quality Inspection
# ============================================================

for name in [
    "challenge",
    "development",
    "locked_test",
    "development_v2"
]:

    df = datasets[name]

    print(f"\n{'=' * 65}")
    print(f"DATASET: {name.upper()}")
    print("=" * 65)

    print("\nCOLUMNS:")
    print(df.columns.tolist())

    print("\nMISSING VALUES:")
    print(df.isna().sum())

    print("\nDATA TYPES:")
    print(df.dtypes)

    print("\nFIRST 3 ROWS:")
    display(df.head(3))

print("\nBENCHMARK QUALITY INSPECTION COMPLETED")


DATASET: CHALLENGE

COLUMNS:
['example_id', 'text', 'expected_category', 'expected_intent', 'test_type', 'challenge_type', 'expected_categories', 'expected_intents', 'requires_clarification', 'annotation_rationale', 'guideline_version']

MISSING VALUES:
example_id                 0
text                       0
expected_category         22
expected_intent           22
test_type                  0
challenge_type             0
expected_categories       15
expected_intents          15
requires_clarification     0
annotation_rationale       0
guideline_version          0
dtype: int64

DATA TYPES:
example_id                    str
text                          str
expected_category         float64
expected_intent           float64
test_type                     str
challenge_type                str
expected_categories           str
expected_intents              str
requires_clarification       bool
annotation_rationale          str
guideline_version         float64
dtype: object

FIRST 3 ROW

,example_id,text,expected_category,expected_intent,test_type,challenge_type,expected_categories,expected_intents,requires_clarification,annotation_rationale,guideline_version
0,CH-001,"I need to make a change to my order, but I'm n...",NaN,NaN,natural,ambiguous,NaN,NaN,True,The customer has not specified whether they wa...,1.3
1,CH-002,I'm having trouble getting into my account. Ca...,NaN,NaN,natural,ambiguous,NaN,NaN,True,The access problem could involve password reco...,1.3
2,CH-003,I want to know what happened to the money from...,NaN,NaN,natural,ambiguous,NaN,NaN,True,It is unclear whether the customer is asking a...,1.3



DATASET: DEVELOPMENT

COLUMNS:
['example_id', 'text', 'expected_category', 'expected_intent', 'test_type', 'challenge_type', 'expected_categories', 'expected_intents', 'requires_clarification', 'annotation_rationale', 'guideline_version']

MISSING VALUES:
example_id                 0
text                       0
expected_category          0
expected_intent            0
test_type                  0
challenge_type            27
expected_categories       27
expected_intents          27
requires_clarification     0
annotation_rationale       0
guideline_version          0
dtype: int64

DATA TYPES:
example_id                    str
text                          str
expected_category             str
expected_intent               str
test_type                     str
challenge_type            float64
expected_categories       float64
expected_intents          float64
requires_clarification       bool
annotation_rationale          str
guideline_version         float64
dtype: object

FIRST 3 R

,example_id,text,expected_category,expected_intent,test_type,challenge_type,expected_categories,expected_intents,requires_clarification,annotation_rationale,guideline_version
0,DEV-001,I need to register a new profile before I can ...,ACCOUNT,create_account,natural,NaN,NaN,NaN,False,The customer wants to create a new account.,1.3
1,DEV-002,The sign-up page keeps rejecting my details ev...,ACCOUNT,registration_problems,contextual,NaN,NaN,NaN,False,The customer reports a problem during account ...,1.3
2,DEV-003,Please close my membership profile permanently,ACCOUNT,delete_account,natural,NaN,NaN,NaN,False,The customer explicitly requests removal of an...,1.3



DATASET: LOCKED_TEST

COLUMNS:
['example_id', 'text', 'expected_category', 'expected_intent', 'test_type', 'challenge_type', 'expected_categories', 'expected_intents', 'requires_clarification', 'annotation_rationale', 'guideline_version']

MISSING VALUES:
example_id                  0
text                        0
expected_category           0
expected_intent             0
test_type                   0
challenge_type            108
expected_categories       108
expected_intents          108
requires_clarification      0
annotation_rationale        0
guideline_version           0
dtype: int64

DATA TYPES:
example_id                    str
text                          str
expected_category             str
expected_intent               str
test_type                     str
challenge_type            float64
expected_categories       float64
expected_intents          float64
requires_clarification       bool
annotation_rationale          str
guideline_version         float64
dtype: object

,example_id,text,expected_category,expected_intent,test_type,challenge_type,expected_categories,expected_intents,requires_clarification,annotation_rationale,guideline_version
0,TST-001,Could you walk me through opening a customer a...,ACCOUNT,create_account,natural,NaN,NaN,NaN,False,The customer wants to create a new account.,1.3
1,TST-002,I'd like to join the platform as a new user; w...,ACCOUNT,create_account,contextual,NaN,NaN,NaN,False,The customer asks how to establish a new user ...,1.3
2,TST-003,I need to register a brand-new account separat...,ACCOUNT,create_account,paraphrase,NaN,NaN,NaN,False,The customer explicitly requests registration ...,1.3



DATASET: DEVELOPMENT_V2

COLUMNS:
['instruction', 'category', 'intent']

MISSING VALUES:
instruction    0
category       0
intent         0
dtype: int64

DATA TYPES:
instruction    str
category       str
intent         str
dtype: object

FIRST 3 ROWS:


,instruction,category,intent
0,I would like to open a new customer account.,ACCOUNT,create_account
1,How can I register for an account on your webs...,ACCOUNT,create_account
2,Please help me create a personal account.,ACCOUNT,create_account



BENCHMARK QUALITY INSPECTION COMPLETED


In [8]:
# ============================================================
# LUMORA — BENCHMARK SUMMARY
# ============================================================

for name in ["development", "locked_test", "development_v2"]:

    df = datasets[name]

    print(f"\n{'=' * 60}")
    print(f"{name.upper()} — {df.shape}")

    print("\nCOLUMNS:")
    print(df.columns.tolist())

    print("\nMISSING VALUES:")
    print(df.isna().sum().to_dict())

    print("\nSAMPLE:")
    display(df.head(2))

    text_col = next(
        (c for c in ["text", "instruction"] if c in df.columns),
        None
    )

    label_col = next(
        (c for c in ["expected_category", "category"] if c in df.columns),
        None
    )

    if text_col and label_col:
        print("\nTEXT COLUMN:", text_col)
        print("LABEL COLUMN:", label_col)
        print("UNIQUE LABELS:", df[label_col].nunique())


DEVELOPMENT — (27, 11)

COLUMNS:
['example_id', 'text', 'expected_category', 'expected_intent', 'test_type', 'challenge_type', 'expected_categories', 'expected_intents', 'requires_clarification', 'annotation_rationale', 'guideline_version']

MISSING VALUES:
{'example_id': 0, 'text': 0, 'expected_category': 0, 'expected_intent': 0, 'test_type': 0, 'challenge_type': 27, 'expected_categories': 27, 'expected_intents': 27, 'requires_clarification': 0, 'annotation_rationale': 0, 'guideline_version': 0}

SAMPLE:


,example_id,text,expected_category,expected_intent,test_type,challenge_type,expected_categories,expected_intents,requires_clarification,annotation_rationale,guideline_version
0,DEV-001,I need to register a new profile before I can ...,ACCOUNT,create_account,natural,NaN,NaN,NaN,False,The customer wants to create a new account.,1.3
1,DEV-002,The sign-up page keeps rejecting my details ev...,ACCOUNT,registration_problems,contextual,NaN,NaN,NaN,False,The customer reports a problem during account ...,1.3



TEXT COLUMN: text
LABEL COLUMN: expected_category
UNIQUE LABELS: 11

LOCKED_TEST — (108, 11)

COLUMNS:
['example_id', 'text', 'expected_category', 'expected_intent', 'test_type', 'challenge_type', 'expected_categories', 'expected_intents', 'requires_clarification', 'annotation_rationale', 'guideline_version']

MISSING VALUES:
{'example_id': 0, 'text': 0, 'expected_category': 0, 'expected_intent': 0, 'test_type': 0, 'challenge_type': 108, 'expected_categories': 108, 'expected_intents': 108, 'requires_clarification': 0, 'annotation_rationale': 0, 'guideline_version': 0}

SAMPLE:


,example_id,text,expected_category,expected_intent,test_type,challenge_type,expected_categories,expected_intents,requires_clarification,annotation_rationale,guideline_version
0,TST-001,Could you walk me through opening a customer a...,ACCOUNT,create_account,natural,NaN,NaN,NaN,False,The customer wants to create a new account.,1.3
1,TST-002,I'd like to join the platform as a new user; w...,ACCOUNT,create_account,contextual,NaN,NaN,NaN,False,The customer asks how to establish a new user ...,1.3



TEXT COLUMN: text
LABEL COLUMN: expected_category
UNIQUE LABELS: 11

DEVELOPMENT_V2 — (135, 3)

COLUMNS:
['instruction', 'category', 'intent']

MISSING VALUES:
{'instruction': 0, 'category': 0, 'intent': 0}

SAMPLE:


,instruction,category,intent
0,I would like to open a new customer account.,ACCOUNT,create_account
1,How can I register for an account on your webs...,ACCOUNT,create_account



TEXT COLUMN: instruction
LABEL COLUMN: category
UNIQUE LABELS: 11


In [9]:
# ============================================================
# LUMORA — PRE-DEPLOYMENT MODEL TESTING
# Step 6: Build Existing Text Reference
# ============================================================

import re

def normalize_text(text):
    """Normalize text for exact duplicate detection."""
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text


existing_texts = {}

text_columns = {
    "training": "instruction",
    "challenge": "text",
    "development": "text",
    "locked_test": "text",
    "development_v2": "instruction",
}

for dataset_name, column in text_columns.items():

    df = datasets[dataset_name]

    normalized = set(
        df[column].dropna().map(normalize_text)
    )

    existing_texts[dataset_name] = normalized

    print(
        f"{dataset_name.upper()}: "
        f"{len(normalized):,} unique normalized texts"
    )


all_existing_texts = set().union(*existing_texts.values())

print("\n" + "=" * 55)
print("TOTAL UNIQUE EXISTING TEXTS:", len(all_existing_texts))
print("REFERENCE SET READY")

TRAINING: 24,011 unique normalized texts
CHALLENGE: 22 unique normalized texts
DEVELOPMENT: 27 unique normalized texts
LOCKED_TEST: 108 unique normalized texts
DEVELOPMENT_V2: 135 unique normalized texts

TOTAL UNIQUE EXISTING TEXTS: 24303
REFERENCE SET READY


In [10]:
# ============================================================
# LUMORA — PRE-DEPLOYMENT MODEL TESTING
# Step 7: Create Classification Test Cases
# ============================================================

classification_cases = [

    # ACCOUNT
    ("PT-001", "I need to update the email linked to my customer profile.", "ACCOUNT", "natural"),
    ("PT-002", "Where can I change the personal details saved in my account?", "ACCOUNT", "paraphrase"),
    ("PT-003", "My profile information is outdated, and I want to correct it.", "ACCOUNT", "contextual"),
    ("PT-004", "I cannot access the profile settings to update my username.", "ACCOUNT", "natural"),

    # CANCEL
    ("PT-005", "Please terminate my current service before the next billing cycle.", "CANCEL", "natural"),
    ("PT-006", "I no longer wish to continue using this service.", "CANCEL", "paraphrase"),
    ("PT-007", "My renewal is approaching, so I want to stop the service beforehand.", "CANCEL", "contextual"),
    ("PT-008", "How can I discontinue an active order or service?", "CANCEL", "natural"),

    # CONTACT
    ("PT-009", "Is there a way to speak directly with a support agent?", "CONTACT", "natural"),
    ("PT-010", "I would prefer to communicate with a real person rather than automated support.", "CONTACT", "paraphrase"),
    ("PT-011", "The automated replies have not resolved my issue, and I need human assistance.", "CONTACT", "contextual"),
    ("PT-012", "What are the available ways to reach your customer service team?", "CONTACT", "natural"),

    # DELIVERY
    ("PT-013", "Can you tell me the estimated arrival date of my package?", "DELIVERY", "natural"),
    ("PT-014", "I need an update on when the courier will bring my purchase.", "DELIVERY", "paraphrase"),
    ("PT-015", "The tracking page has not changed since yesterday, and I am waiting for the parcel.", "DELIVERY", "contextual"),
    ("PT-016", "When should I expect my shipment to reach my home?", "DELIVERY", "natural"),

    # FEEDBACK
    ("PT-017", "I would like to share my experience with your customer service.", "FEEDBACK", "natural"),
    ("PT-018", "Where can I leave comments about the service I received?", "FEEDBACK", "paraphrase"),
    ("PT-019", "After completing my purchase, I noticed something I would like to suggest improving.", "FEEDBACK", "contextual"),
    ("PT-020", "I want to submit a review about my recent experience.", "FEEDBACK", "natural"),

    # INVOICE
    ("PT-021", "Could you provide a copy of the invoice for my latest purchase?", "INVOICE", "natural"),
    ("PT-022", "Where can I download the billing document associated with my transaction?", "INVOICE", "paraphrase"),
    ("PT-023", "My finance department needs the purchase invoice for record keeping.", "INVOICE", "contextual"),
    ("PT-024", "I cannot locate the receipt document in my account.", "INVOICE", "natural"),

    # ORDER
    ("PT-025", "I want to know whether my purchase has been confirmed.", "ORDER", "natural"),
    ("PT-026", "Can I review the details of the order I placed earlier?", "ORDER", "paraphrase"),
    ("PT-027", "I completed checkout, but no confirmation appeared on the screen.", "ORDER", "contextual"),
    ("PT-028", "Where can I find the reference number for my recent order?", "ORDER", "natural"),

    # PAYMENT
    ("PT-029", "Which payment options can I use at checkout?", "PAYMENT", "natural"),
    ("PT-030", "I need to replace the card used for future transactions.", "PAYMENT", "paraphrase"),
    ("PT-031", "The checkout page rejected my card even though it is still valid.", "PAYMENT", "contextual"),
    ("PT-032", "How can I verify whether my transaction was successfully charged?", "PAYMENT", "natural"),

    # REFUND
    ("PT-033", "I want to request my money back for a returned item.", "REFUND", "natural"),
    ("PT-034", "How long does it usually take for a reimbursement to arrive?", "REFUND", "paraphrase"),
    ("PT-035", "The returned product was received, but I have not received the money back.", "REFUND", "contextual"),
    ("PT-036", "Please explain how I can claim a refund for my purchase.", "REFUND", "natural"),

    # SHIPPING
    ("PT-037", "Can I have my package sent to a different destination?", "SHIPPING", "natural"),
    ("PT-038", "I need to modify the address used for sending my purchases.", "SHIPPING", "paraphrase"),
    ("PT-039", "I recently moved, and future parcels should go to my new location.", "SHIPPING", "contextual"),
    ("PT-040", "Do you offer delivery to international addresses?", "SHIPPING", "natural"),

    # SUBSCRIPTION
    ("PT-041", "How can I change the frequency of your email updates?", "SUBSCRIPTION", "natural"),
    ("PT-042", "I want to receive your newsletter only once a month.", "SUBSCRIPTION", "paraphrase"),
    ("PT-043", "The promotional emails arrive too frequently, and I want to adjust my preferences.", "SUBSCRIPTION", "contextual"),
    ("PT-044", "Where can I manage the emails I am subscribed to?", "SUBSCRIPTION", "natural"),
]

classification_test = pd.DataFrame(
    classification_cases,
    columns=["example_id", "text", "expected_category", "test_type"]
)

# Normalize new test texts
classification_test["normalized_text"] = (
    classification_test["text"].map(normalize_text)
)

# Check exact normalized duplicates inside the new set
internal_duplicates = classification_test[
    classification_test["normalized_text"].duplicated(keep=False)
]

# Check overlap with all previous datasets
overlap_mask = classification_test["normalized_text"].isin(
    all_existing_texts
)

overlaps = classification_test[overlap_mask]

# Category distribution
print("TOTAL TEST CASES:", len(classification_test))

print("\nCATEGORY DISTRIBUTION:")
print(classification_test["expected_category"].value_counts().sort_index())

print("\nTEST TYPE DISTRIBUTION:")
print(classification_test["test_type"].value_counts())

print("\nINTERNAL DUPLICATES:", len(internal_duplicates))
print("OVERLAP WITH EXISTING DATA:", len(overlaps))

if len(internal_duplicates) > 0:
    display(internal_duplicates[["example_id", "text"]])

if len(overlaps) > 0:
    display(overlaps[["example_id", "text"]])

print("\nCLASSIFICATION TEST SET CREATED.")

TOTAL TEST CASES: 44

CATEGORY DISTRIBUTION:
expected_category
ACCOUNT         4
CANCEL          4
CONTACT         4
DELIVERY        4
FEEDBACK        4
INVOICE         4
ORDER           4
PAYMENT         4
REFUND          4
SHIPPING        4
SUBSCRIPTION    4
Name: count, dtype: int64

TEST TYPE DISTRIBUTION:
test_type
natural       22
paraphrase    11
contextual    11
Name: count, dtype: int64

INTERNAL DUPLICATES: 0
OVERLAP WITH EXISTING DATA: 0

CLASSIFICATION TEST SET CREATED.


In [11]:
# ============================================================
# LUMORA — SEMANTIC REVIEW
# Refine two potentially ambiguous cases
# ============================================================

classification_test.loc[
    classification_test["example_id"] == "PT-008",
    "text"
] = "I would like to cancel the order I placed earlier today."

classification_test.loc[
    classification_test["example_id"] == "PT-024",
    "text"
] = "I need a copy of the tax invoice issued for my purchase."

# Refresh normalized text
classification_test["normalized_text"] = (
    classification_test["text"].map(normalize_text)
)

# Recheck duplicates and overlap
internal_duplicates = classification_test[
    classification_test["normalized_text"].duplicated(keep=False)
]

overlaps = classification_test[
    classification_test["normalized_text"].isin(all_existing_texts)
]

print("TOTAL CASES:", len(classification_test))
print("INTERNAL DUPLICATES:", len(internal_duplicates))
print("EXISTING DATA OVERLAP:", len(overlaps))

print("\nUPDATED CASES:")
display(
    classification_test[
        classification_test["example_id"].isin(["PT-008", "PT-024"])
    ][["example_id", "text", "expected_category"]]
)

TOTAL CASES: 44
INTERNAL DUPLICATES: 0
EXISTING DATA OVERLAP: 0

UPDATED CASES:


,example_id,text,expected_category
7,PT-008,I would like to cancel the order I placed earl...,CANCEL
23,PT-024,I need a copy of the tax invoice issued for my...,INVOICE


In [12]:
# ============================================================
# LUMORA — PRE-DEPLOYMENT MODEL TESTING
# Step 9: Create Ambiguous Test Cases
# ============================================================

ambiguous_cases = [

    (
        "AMB-001",
        "I need help with my order.",
        ["ORDER", "DELIVERY"],
        "The customer does not explain whether the issue concerns order details or delivery."
    ),
    (
        "AMB-002",
        "There is a problem with the money from my purchase.",
        ["PAYMENT", "REFUND"],
        "It is unclear whether the customer has a payment issue or wants money returned."
    ),
    (
        "AMB-003",
        "I cannot get into my account and need someone to help.",
        ["ACCOUNT", "CONTACT"],
        "The customer may need account access assistance or human support."
    ),
    (
        "AMB-004",
        "I want to stop receiving this service.",
        ["CANCEL", "SUBSCRIPTION"],
        "The customer does not specify whether this means cancelling a service or managing a subscription."
    ),
    (
        "AMB-005",
        "I need something showing what I paid.",
        ["INVOICE", "PAYMENT"],
        "The customer may want an invoice or information about a payment."
    ),
    (
        "AMB-006",
        "My package is not where I expected it to be.",
        ["DELIVERY", "SHIPPING"],
        "It is unclear whether the customer is asking about delivery progress or shipping arrangements."
    ),
    (
        "AMB-007",
        "I want to change something about my purchase.",
        ["ORDER", "PAYMENT"],
        "The requested change could concern order details or payment information."
    ),
    (
        "AMB-008",
        "I am unhappy with what happened after my purchase.",
        ["FEEDBACK", "REFUND"],
        "The customer expresses dissatisfaction but does not specify whether they want to provide feedback or request a refund."
    ),
    (
        "AMB-009",
        "I need to update my details before the next order.",
        ["ACCOUNT", "SHIPPING"],
        "The details could refer to account information or a delivery address."
    ),
    (
        "AMB-010",
        "I want to receive fewer messages from your company.",
        ["SUBSCRIPTION", "FEEDBACK"],
        "The customer may want to change communication preferences or provide feedback."
    ),
    (
        "AMB-011",
        "I need to talk to someone about a charge on my account.",
        ["CONTACT", "PAYMENT"],
        "The customer requests human assistance but also mentions a payment-related issue."
    ),
]

ambiguous_test = pd.DataFrame(
    ambiguous_cases,
    columns=[
        "example_id",
        "text",
        "expected_categories",
        "annotation_rationale"
    ]
)

ambiguous_test["requires_clarification"] = True
ambiguous_test["normalized_text"] = (
    ambiguous_test["text"].map(normalize_text)
)

# Duplicate checks
amb_internal_duplicates = ambiguous_test[
    ambiguous_test["normalized_text"].duplicated(keep=False)
]

amb_existing_overlap = ambiguous_test[
    ambiguous_test["normalized_text"].isin(all_existing_texts)
]

classification_overlap = ambiguous_test[
    ambiguous_test["normalized_text"].isin(
        set(classification_test["normalized_text"])
    )
]

print("AMBIGUOUS CASES:", len(ambiguous_test))
print("INTERNAL DUPLICATES:", len(amb_internal_duplicates))
print("OVERLAP WITH PREVIOUS DATA:", len(amb_existing_overlap))
print("OVERLAP WITH CLASSIFICATION SET:", len(classification_overlap))

display(
    ambiguous_test[
        ["example_id", "text", "expected_categories", "requires_clarification"]
    ]
)

AMBIGUOUS CASES: 11
INTERNAL DUPLICATES: 0
OVERLAP WITH PREVIOUS DATA: 0
OVERLAP WITH CLASSIFICATION SET: 0


,example_id,text,expected_categories,requires_clarification
0,AMB-001,I need help with my order.,"[ORDER, DELIVERY]",True
1,AMB-002,There is a problem with the money from my purc...,"[PAYMENT, REFUND]",True
2,AMB-003,I cannot get into my account and need someone ...,"[ACCOUNT, CONTACT]",True
3,AMB-004,I want to stop receiving this service.,"[CANCEL, SUBSCRIPTION]",True
4,AMB-005,I need something showing what I paid.,"[INVOICE, PAYMENT]",True
5,AMB-006,My package is not where I expected it to be.,"[DELIVERY, SHIPPING]",True
6,AMB-007,I want to change something about my purchase.,"[ORDER, PAYMENT]",True
7,AMB-008,I am unhappy with what happened after my purch...,"[FEEDBACK, REFUND]",True
8,AMB-009,I need to update my details before the next or...,"[ACCOUNT, SHIPPING]",True
9,AMB-010,I want to receive fewer messages from your com...,"[SUBSCRIPTION, FEEDBACK]",True


In [13]:
# ============================================================
# LUMORA — PRE-DEPLOYMENT MODEL TESTING
# Step 10: Create Out-of-Scope Test Cases
# ============================================================

out_of_scope_cases = [

    ("OOS-001", "What will the weather be like in Riyadh tomorrow?", "Weather inquiry"),
    ("OOS-002", "Can you explain how neural networks learn?", "General knowledge"),
    ("OOS-003", "I need a recipe for homemade chocolate cake.", "Cooking request"),
    ("OOS-004", "Who won the latest football championship?", "Sports inquiry"),
    ("OOS-005", "Can you translate this paragraph into Arabic?", "Translation request"),
    ("OOS-006", "Write a short birthday message for my friend.", "Creative writing"),
    ("OOS-007", "What is the capital city of Japan?", "General knowledge"),
    ("OOS-008", "Help me calculate the average of these numbers.", "Mathematical request"),
    ("OOS-009", "Recommend some books about data science.", "Recommendation request"),
    ("OOS-010", "Explain the difference between Python and Java.", "Programming knowledge"),
]

out_of_scope_test = pd.DataFrame(
    out_of_scope_cases,
    columns=["example_id", "text", "out_of_scope_type"]
)

out_of_scope_test["normalized_text"] = (
    out_of_scope_test["text"].map(normalize_text)
)

# Duplicate checks
oos_internal_duplicates = out_of_scope_test[
    out_of_scope_test["normalized_text"].duplicated(keep=False)
]

oos_existing_overlap = out_of_scope_test[
    out_of_scope_test["normalized_text"].isin(all_existing_texts)
]

oos_classification_overlap = out_of_scope_test[
    out_of_scope_test["normalized_text"].isin(
        set(classification_test["normalized_text"])
    )
]

oos_ambiguous_overlap = out_of_scope_test[
    out_of_scope_test["normalized_text"].isin(
        set(ambiguous_test["normalized_text"])
    )
]

print("OUT-OF-SCOPE CASES:", len(out_of_scope_test))
print("INTERNAL DUPLICATES:", len(oos_internal_duplicates))
print("OVERLAP WITH PREVIOUS DATA:", len(oos_existing_overlap))
print("OVERLAP WITH CLASSIFICATION:", len(oos_classification_overlap))
print("OVERLAP WITH AMBIGUOUS:", len(oos_ambiguous_overlap))

display(out_of_scope_test[["example_id", "text", "out_of_scope_type"]])

OUT-OF-SCOPE CASES: 10
INTERNAL DUPLICATES: 0
OVERLAP WITH PREVIOUS DATA: 0
OVERLAP WITH CLASSIFICATION: 0
OVERLAP WITH AMBIGUOUS: 0


,example_id,text,out_of_scope_type
0,OOS-001,What will the weather be like in Riyadh tomorrow?,Weather inquiry
1,OOS-002,Can you explain how neural networks learn?,General knowledge
2,OOS-003,I need a recipe for homemade chocolate cake.,Cooking request
3,OOS-004,Who won the latest football championship?,Sports inquiry
4,OOS-005,Can you translate this paragraph into Arabic?,Translation request
5,OOS-006,Write a short birthday message for my friend.,Creative writing
6,OOS-007,What is the capital city of Japan?,General knowledge
7,OOS-008,Help me calculate the average of these numbers.,Mathematical request
8,OOS-009,Recommend some books about data science.,Recommendation request
9,OOS-010,Explain the difference between Python and Java.,Programming knowledge


In [14]:
# ============================================================
# LUMORA — PRE-DEPLOYMENT TEST SUITE
# Combine and validate all independent test cases
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. Standardize classification cases
# ------------------------------------------------------------

classification_records = []

for _, row in classification_test.iterrows():
    classification_records.append({
        "example_id": row["example_id"],
        "text": row["text"],
        "test_group": "classification",
        "test_type": row["test_type"],
        "expected_category": row["expected_category"],
        "expected_categories": None,
        "is_in_scope": True,
        "requires_clarification": False,
        "out_of_scope_type": None,
        "annotation_rationale": None,
    })


# ------------------------------------------------------------
# 2. Standardize ambiguous cases
# ------------------------------------------------------------

ambiguous_records = []

for _, row in ambiguous_test.iterrows():
    accepted_categories = row["expected_categories"]

    if isinstance(accepted_categories, (list, tuple, set)):
        accepted_categories = "|".join(sorted(accepted_categories))

    ambiguous_records.append({
        "example_id": row["example_id"],
        "text": row["text"],
        "test_group": "ambiguous",
        "test_type": "ambiguous",
        "expected_category": None,
        "expected_categories": accepted_categories,
        "is_in_scope": True,
        "requires_clarification": True,
        "out_of_scope_type": None,
        "annotation_rationale": row.get("annotation_rationale", None),
    })


# ------------------------------------------------------------
# 3. Standardize out-of-scope cases
# ------------------------------------------------------------

out_of_scope_records = []

for _, row in out_of_scope_test.iterrows():
    out_of_scope_records.append({
        "example_id": row["example_id"],
        "text": row["text"],
        "test_group": "out_of_scope",
        "test_type": "out_of_scope",
        "expected_category": None,
        "expected_categories": None,
        "is_in_scope": False,
        "requires_clarification": None,
        "out_of_scope_type": row["out_of_scope_type"],
        "annotation_rationale": None,
    })


# ------------------------------------------------------------
# 4. Combine all test groups
# ------------------------------------------------------------

predeployment_test = pd.DataFrame(
    classification_records
    + ambiguous_records
    + out_of_scope_records
)

# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

predeployment_test["normalized_text"] = (
    predeployment_test["text"].map(normalize_text)
)

print("TOTAL TEST CASES:", len(predeployment_test))
print("\nTEST GROUP DISTRIBUTION:")
display(predeployment_test["test_group"].value_counts())

print("\nDUPLICATE IDs:")
print(predeployment_test["example_id"].duplicated().sum())

print("\nDUPLICATE NORMALIZED TEXTS:")
print(predeployment_test["normalized_text"].duplicated().sum())

print("\nMISSING TEXT VALUES:")
print(predeployment_test["text"].isna().sum())

print("\nMISSING IDs:")
print(predeployment_test["example_id"].isna().sum())

# ------------------------------------------------------------
# 6. Save the independent test suite
# ------------------------------------------------------------

output_path = (
    PROJECT_ROOT
    / "data"
    / "benchmark"
    / "predeployment_test_v1.csv"
)

predeployment_test.drop(
    columns=["normalized_text"]
).to_csv(output_path, index=False)

print("\nSAVED TEST SUITE:")
print(output_path)

print("\nFILE EXISTS:", output_path.exists())
print("FILE SIZE:", round(output_path.stat().st_size / 1024, 2), "KB")

TOTAL TEST CASES: 65

TEST GROUP DISTRIBUTION:


test_group
classification    44
ambiguous         11
out_of_scope      10
Name: count, dtype: int64


DUPLICATE IDs:
0

DUPLICATE NORMALIZED TEXTS:
0

MISSING TEXT VALUES:
0

MISSING IDs:
0

SAVED TEST SUITE:
/Users/raghad/Desktop/Lumora/data/benchmark/predeployment_test_v1.csv

FILE EXISTS: True
FILE SIZE: 8.19 KB


In [15]:
# ============================================================
# LUMORA — PRE-DEPLOYMENT TEST 01
# Classification Performance
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# ------------------------------------------------------------
# 1. Load the saved independent test suite
# ------------------------------------------------------------

test_path = (
    PROJECT_ROOT
    / "data"
    / "benchmark"
    / "predeployment_test_v1.csv"
)

test_suite = pd.read_csv(test_path)

classification_eval = test_suite[
    test_suite["test_group"] == "classification"
].copy()

X_test = classification_eval["text"]
y_true = classification_eval["expected_category"]

print("TEST CASES:", len(classification_eval))
print("UNIQUE EXPECTED CLASSES:", y_true.nunique())


# ------------------------------------------------------------
# 2. Generate predictions using the saved model
# ------------------------------------------------------------

y_pred = model.predict(X_test)

# Model probability scores
y_proba = model.predict_proba(X_test)

class_names = model.classes_

top_indices = np.argsort(y_proba, axis=1)[:, ::-1]

top_predictions = [
    class_names[indices[0]]
    for indices in top_indices
]

second_predictions = [
    class_names[indices[1]]
    for indices in top_indices
]

top_scores = y_proba.max(axis=1)

second_scores = np.partition(
    y_proba,
    -2,
    axis=1
)[:, -2]

prediction_margin = top_scores - second_scores


# ------------------------------------------------------------
# 3. Calculate overall metrics
# ------------------------------------------------------------

accuracy = accuracy_score(y_true, y_pred)

balanced_accuracy = balanced_accuracy_score(
    y_true,
    y_pred
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("\n===== OVERALL PERFORMANCE =====")

print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced Accuracy: {balanced_accuracy:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")


# ------------------------------------------------------------
# 4. Per-class performance
# ------------------------------------------------------------

report = classification_report(
    y_true,
    y_pred,
    labels=class_names,
    output_dict=True,
    zero_division=0
)

class_report = pd.DataFrame(report).T

print("\n===== PER-CLASS PERFORMANCE =====")

display(
    class_report.loc[
        list(class_names),
        ["precision", "recall", "f1-score", "support"]
    ].round(3)
)


# ------------------------------------------------------------
# 5. Build detailed prediction results
# ------------------------------------------------------------

classification_results = classification_eval[
    ["example_id", "text", "expected_category", "test_type"]
].copy()

classification_results["predicted_category"] = y_pred

classification_results["correct"] = (
    classification_results["expected_category"]
    == classification_results["predicted_category"]
)

classification_results["top_score"] = top_scores

classification_results["second_prediction"] = second_predictions

classification_results["second_score"] = second_scores

classification_results["prediction_margin"] = prediction_margin

print("\n===== PREDICTION RESULTS =====")

display(
    classification_results[
        [
            "example_id",
            "expected_category",
            "predicted_category",
            "correct",
            "top_score",
            "second_prediction",
            "prediction_margin",
        ]
    ].round(3)
)


# ------------------------------------------------------------
# 6. Analyze performance by test type
# ------------------------------------------------------------

test_type_results = (
    classification_results
    .groupby("test_type")
    .agg(
        total_cases=("correct", "size"),
        correct_predictions=("correct", "sum"),
        accuracy=("correct", "mean"),
    )
)

test_type_results["accuracy"] *= 100

print("\n===== PERFORMANCE BY TEST TYPE =====")

display(test_type_results.round(2))


# ------------------------------------------------------------
# 7. Inspect classification errors
# ------------------------------------------------------------

errors = classification_results[
    ~classification_results["correct"]
].copy()

print("\n===== MISCLASSIFIED CASES =====")
print("TOTAL ERRORS:", len(errors))

display(
    errors[
        [
            "example_id",
            "text",
            "expected_category",
            "predicted_category",
            "test_type",
            "top_score",
            "second_prediction",
            "prediction_margin",
        ]
    ].round(3)
)


# ------------------------------------------------------------
# 8. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=class_names
)

confusion_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

print("\n===== CONFUSION MATRIX =====")

display(confusion_df)

TEST CASES: 44
UNIQUE EXPECTED CLASSES: 11

===== OVERALL PERFORMANCE =====
Accuracy:          0.6136
Balanced Accuracy: 0.6136
Macro F1:          0.5970
Weighted F1:       0.5970

===== PER-CLASS PERFORMANCE =====


,precision,recall,f1-score,support
ACCOUNT,0.286,1.00,0.444,4.0
CANCEL,0.000,0.00,0.000,4.0
CONTACT,0.600,0.75,0.667,4.0
DELIVERY,0.500,0.50,0.500,4.0
FEEDBACK,1.000,0.50,0.667,4.0
INVOICE,1.000,1.00,1.000,4.0
ORDER,0.600,0.75,0.667,4.0
PAYMENT,1.000,0.25,0.400,4.0
REFUND,0.800,1.00,0.889,4.0
SHIPPING,1.000,0.50,0.667,4.0



===== PREDICTION RESULTS =====


,example_id,expected_category,predicted_category,correct,top_score,second_prediction,prediction_margin
0,PT-001,ACCOUNT,ACCOUNT,True,0.531,CONTACT,0.349
1,PT-002,ACCOUNT,ACCOUNT,True,0.972,SHIPPING,0.967
2,PT-003,ACCOUNT,ACCOUNT,True,0.857,SHIPPING,0.814
3,PT-004,ACCOUNT,ACCOUNT,True,0.912,SHIPPING,0.875
4,PT-005,CANCEL,ACCOUNT,False,0.191,CANCEL,0.025
5,PT-006,CANCEL,CONTACT,False,0.292,ACCOUNT,0.060
6,PT-007,CANCEL,ACCOUNT,False,0.189,REFUND,0.042
7,PT-008,CANCEL,ORDER,False,0.621,ACCOUNT,0.527
8,PT-009,CONTACT,CONTACT,True,0.964,FEEDBACK,0.954
9,PT-010,CONTACT,CONTACT,True,0.834,ACCOUNT,0.782



===== PERFORMANCE BY TEST TYPE =====


,total_cases,correct_predictions,accuracy
test_type,,,
contextual,11,3,27.27
natural,22,15,68.18
paraphrase,11,9,81.82



===== MISCLASSIFIED CASES =====
TOTAL ERRORS: 17


,example_id,text,expected_category,predicted_category,test_type,top_score,second_prediction,prediction_margin
4,PT-005,Please terminate my current service before the...,CANCEL,ACCOUNT,natural,0.191,CANCEL,0.025
5,PT-006,I no longer wish to continue using this service.,CANCEL,CONTACT,paraphrase,0.292,ACCOUNT,0.060
6,PT-007,"My renewal is approaching, so I want to stop t...",CANCEL,ACCOUNT,contextual,0.189,REFUND,0.042
7,PT-008,I would like to cancel the order I placed earl...,CANCEL,ORDER,natural,0.621,ACCOUNT,0.527
10,PT-011,The automated replies have not resolved my iss...,CONTACT,ACCOUNT,contextual,0.390,CONTACT,0.259
12,PT-013,Can you tell me the estimated arrival date of ...,DELIVERY,ACCOUNT,natural,0.266,DELIVERY,0.037
14,PT-015,The tracking page has not changed since yester...,DELIVERY,REFUND,contextual,0.196,ACCOUNT,0.009
16,PT-017,I would like to share my experience with your ...,FEEDBACK,CONTACT,natural,0.512,FEEDBACK,0.132
18,PT-019,"After completing my purchase, I noticed someth...",FEEDBACK,ORDER,contextual,0.536,ACCOUNT,0.425
26,PT-027,"I completed checkout, but no confirmation appe...",ORDER,ACCOUNT,contextual,0.273,REFUND,0.110



===== CONFUSION MATRIX =====


,ACCOUNT,CANCEL,CONTACT,DELIVERY,FEEDBACK,INVOICE,ORDER,PAYMENT,REFUND,SHIPPING,SUBSCRIPTION
ACCOUNT,4,0,0,0,0,0,0,0,0,0,0
CANCEL,2,0,1,0,0,0,1,0,0,0,0
CONTACT,1,0,3,0,0,0,0,0,0,0,0
DELIVERY,1,0,0,2,0,0,0,0,1,0,0
FEEDBACK,0,0,1,0,2,0,1,0,0,0,0
INVOICE,0,0,0,0,0,4,0,0,0,0,0
ORDER,1,0,0,0,0,0,3,0,0,0,0
PAYMENT,3,0,0,0,0,0,0,1,0,0,0
REFUND,0,0,0,0,0,0,0,0,4,0,0
SHIPPING,1,0,0,1,0,0,0,0,0,2,0


In [16]:
# ============================================================
# SAVE CLASSIFICATION TEST RESULTS
# ============================================================

results_dir = PROJECT_ROOT / "data" / "benchmark"

# Save detailed predictions
classification_results_path = (
    results_dir / "classification_results_v1.csv"
)

classification_results.to_csv(
    classification_results_path,
    index=False
)

# Save per-class metrics
class_metrics_path = (
    results_dir / "classification_metrics_v1.csv"
)

class_report.loc[
    list(class_names),
    ["precision", "recall", "f1-score", "support"]
].to_csv(class_metrics_path)

# Save overall metrics
overall_metrics = pd.DataFrame([{
    "test_cases": len(classification_eval),
    "correct_predictions": int(
        classification_results["correct"].sum()
    ),
    "errors": len(errors),
    "accuracy": accuracy,
    "balanced_accuracy": balanced_accuracy,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
}])

overall_metrics_path = (
    results_dir / "classification_overall_metrics_v1.csv"
)

overall_metrics.to_csv(
    overall_metrics_path,
    index=False
)

print("CLASSIFICATION RESULTS SAVED")

for path in [
    classification_results_path,
    class_metrics_path,
    overall_metrics_path,
]:
    print(path.name, "|", path.exists())

CLASSIFICATION RESULTS SAVED
classification_results_v1.csv | True
classification_metrics_v1.csv | True
classification_overall_metrics_v1.csv | True


In [17]:
# ============================================================
# LUMORA — PRE-DEPLOYMENT TEST 02
# Ambiguous Cases Analysis
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Select ambiguous cases
# ------------------------------------------------------------

ambiguous_eval = test_suite[
    test_suite["test_group"] == "ambiguous"
].copy()

X_ambiguous = ambiguous_eval["text"]

print("AMBIGUOUS TEST CASES:", len(ambiguous_eval))


# ------------------------------------------------------------
# 2. Generate predictions
# ------------------------------------------------------------

ambiguous_predictions = model.predict(X_ambiguous)

ambiguous_proba = model.predict_proba(X_ambiguous)

class_names = model.classes_

top_indices = np.argsort(
    ambiguous_proba,
    axis=1
)[:, ::-1]

top_scores = ambiguous_proba.max(axis=1)

second_scores = np.partition(
    ambiguous_proba,
    -2,
    axis=1
)[:, -2]

prediction_margins = top_scores - second_scores

second_predictions = [
    class_names[indices[1]]
    for indices in top_indices
]


# ------------------------------------------------------------
# 3. Build analysis table
# ------------------------------------------------------------

ambiguous_results = ambiguous_eval[
    [
        "example_id",
        "text",
        "expected_categories",
        "requires_clarification",
        "annotation_rationale",
    ]
].copy()

ambiguous_results["predicted_category"] = (
    ambiguous_predictions
)

ambiguous_results["top_score"] = top_scores

ambiguous_results["second_prediction"] = (
    second_predictions
)

ambiguous_results["second_score"] = second_scores

ambiguous_results["prediction_margin"] = (
    prediction_margins
)


# ------------------------------------------------------------
# 4. Check whether top predictions match accepted categories
# ------------------------------------------------------------

def parse_expected_categories(value):
    if pd.isna(value):
        return []

    return [
        category.strip()
        for category in str(value).split("|")
    ]


ambiguous_results["accepted_categories"] = (
    ambiguous_results["expected_categories"]
    .map(parse_expected_categories)
)

ambiguous_results["top1_matches_accepted"] = (
    ambiguous_results.apply(
        lambda row: (
            row["predicted_category"]
            in row["accepted_categories"]
        ),
        axis=1
    )
)

ambiguous_results["top2_includes_accepted"] = (
    ambiguous_results.apply(
        lambda row: (
            row["predicted_category"]
            in row["accepted_categories"]
            or row["second_prediction"]
            in row["accepted_categories"]
        ),
        axis=1
    )
)


# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

print("\n===== AMBIGUOUS CASE PREDICTIONS =====")

display(
    ambiguous_results[
        [
            "example_id",
            "expected_categories",
            "predicted_category",
            "top_score",
            "second_prediction",
            "second_score",
            "prediction_margin",
            "top1_matches_accepted",
            "top2_includes_accepted",
        ]
    ].round(3)
)


# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------

print("\n===== SUMMARY =====")

print(
    "Top-1 prediction within accepted categories:",
    ambiguous_results["top1_matches_accepted"].sum(),
    "/",
    len(ambiguous_results)
)

print(
    "At least one top-2 prediction within accepted categories:",
    ambiguous_results["top2_includes_accepted"].sum(),
    "/",
    len(ambiguous_results)
)

print(
    "Average top score:",
    round(ambiguous_results["top_score"].mean(), 4)
)

print(
    "Average prediction margin:",
    round(ambiguous_results["prediction_margin"].mean(), 4)
)


# ------------------------------------------------------------
# 7. Inspect probability distributions
# ------------------------------------------------------------

probability_columns = pd.DataFrame(
    ambiguous_proba,
    columns=class_names,
    index=ambiguous_results["example_id"]
)

print("\n===== FULL CLASS PROBABILITIES =====")

display(probability_columns.round(3))

AMBIGUOUS TEST CASES: 11

===== AMBIGUOUS CASE PREDICTIONS =====


,example_id,expected_categories,predicted_category,top_score,second_prediction,second_score,prediction_margin,top1_matches_accepted,top2_includes_accepted
44,AMB-001,DELIVERY|ORDER,ORDER,0.440,ACCOUNT,0.235,0.205,True,True
45,AMB-002,PAYMENT|REFUND,ACCOUNT,0.291,REFUND,0.208,0.084,False,True
46,AMB-003,ACCOUNT|CONTACT,ACCOUNT,0.672,CONTACT,0.148,0.524,True,True
47,AMB-004,CANCEL|SUBSCRIPTION,CONTACT,0.205,REFUND,0.192,0.013,False,False
48,AMB-005,INVOICE|PAYMENT,ORDER,0.430,CONTACT,0.114,0.316,False,False
49,AMB-006,DELIVERY|SHIPPING,DELIVERY,0.453,ACCOUNT,0.113,0.341,True,True
50,AMB-007,ORDER|PAYMENT,ORDER,0.608,ACCOUNT,0.152,0.456,True,True
51,AMB-008,FEEDBACK|REFUND,ACCOUNT,0.231,ORDER,0.189,0.042,False,False
52,AMB-009,ACCOUNT|SHIPPING,ACCOUNT,0.481,ORDER,0.190,0.291,True,True
53,AMB-010,FEEDBACK|SUBSCRIPTION,SUBSCRIPTION,0.367,FEEDBACK,0.178,0.189,True,True



===== SUMMARY =====
Top-1 prediction within accepted categories: 7 / 11
At least one top-2 prediction within accepted categories: 8 / 11
Average top score: 0.4322
Average prediction margin: 0.2453

===== FULL CLASS PROBABILITIES =====


,ACCOUNT,CANCEL,CONTACT,DELIVERY,FEEDBACK,INVOICE,ORDER,PAYMENT,REFUND,SHIPPING,SUBSCRIPTION
example_id,,,,,,,,,,,
AMB-001,0.235,0.016,0.072,0.055,0.034,0.026,0.440,0.043,0.040,0.026,0.014
AMB-002,0.291,0.026,0.033,0.102,0.026,0.074,0.115,0.048,0.208,0.060,0.017
AMB-003,0.672,0.010,0.148,0.013,0.018,0.034,0.023,0.016,0.035,0.021,0.010
AMB-004,0.089,0.030,0.205,0.045,0.157,0.042,0.065,0.039,0.192,0.038,0.097
AMB-005,0.067,0.027,0.114,0.081,0.047,0.042,0.430,0.049,0.088,0.036,0.019
AMB-006,0.113,0.021,0.054,0.453,0.048,0.054,0.050,0.039,0.104,0.047,0.018
AMB-007,0.152,0.010,0.019,0.049,0.033,0.016,0.608,0.013,0.016,0.069,0.014
AMB-008,0.231,0.028,0.113,0.105,0.110,0.048,0.189,0.052,0.063,0.040,0.022
AMB-009,0.481,0.023,0.013,0.033,0.017,0.031,0.190,0.014,0.033,0.139,0.027


In [18]:
# ============================================================
# SAVE AMBIGUOUS TEST RESULTS
# ============================================================

results_dir = PROJECT_ROOT / "data" / "benchmark"

# 1. Detailed results
ambiguous_results_path = (
    results_dir / "ambiguous_results_v1.csv"
)

ambiguous_results.to_csv(
    ambiguous_results_path,
    index=False
)

# 2. Full probability distribution
ambiguous_probabilities_path = (
    results_dir / "ambiguous_probabilities_v1.csv"
)

probability_columns.to_csv(
    ambiguous_probabilities_path
)

# 3. Summary metrics
ambiguous_summary = pd.DataFrame([{
    "test_cases": len(ambiguous_results),
    "top1_matches_accepted": int(
        ambiguous_results["top1_matches_accepted"].sum()
    ),
    "top2_includes_accepted": int(
        ambiguous_results["top2_includes_accepted"].sum()
    ),
    "top1_match_rate": (
        ambiguous_results["top1_matches_accepted"].mean()
    ),
    "top2_match_rate": (
        ambiguous_results["top2_includes_accepted"].mean()
    ),
    "average_top_score": (
        ambiguous_results["top_score"].mean()
    ),
    "average_prediction_margin": (
        ambiguous_results["prediction_margin"].mean()
    ),
}])

ambiguous_summary_path = (
    results_dir / "ambiguous_summary_v1.csv"
)

ambiguous_summary.to_csv(
    ambiguous_summary_path,
    index=False
)

print("AMBIGUOUS TEST RESULTS SAVED")

for path in [
    ambiguous_results_path,
    ambiguous_probabilities_path,
    ambiguous_summary_path,
]:
    print(path.name, "|", path.exists())

AMBIGUOUS TEST RESULTS SAVED
ambiguous_results_v1.csv | True
ambiguous_probabilities_v1.csv | True
ambiguous_summary_v1.csv | True


In [19]:
# ============================================================
# LUMORA — PRE-DEPLOYMENT TEST 03
# Out-of-Scope Detection
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Select out-of-scope cases
# ------------------------------------------------------------

oos_eval = test_suite[
    test_suite["test_group"] == "out_of_scope"
].copy()

X_oos = oos_eval["text"]

print("OUT-OF-SCOPE TEST CASES:", len(oos_eval))


# ------------------------------------------------------------
# 2. Generate predictions
# ------------------------------------------------------------

oos_predictions = model.predict(X_oos)

oos_proba = model.predict_proba(X_oos)

class_names = model.classes_

top_indices = np.argsort(
    oos_proba,
    axis=1
)[:, ::-1]

top_scores = oos_proba.max(axis=1)

second_scores = np.partition(
    oos_proba,
    -2,
    axis=1
)[:, -2]

prediction_margins = top_scores - second_scores

second_predictions = [
    class_names[indices[1]]
    for indices in top_indices
]


# ------------------------------------------------------------
# 3. Build results table
# ------------------------------------------------------------

oos_results = oos_eval[
    [
        "example_id",
        "text",
        "out_of_scope_type",
    ]
].copy()

oos_results["predicted_category"] = oos_predictions

oos_results["top_score"] = top_scores

oos_results["second_prediction"] = second_predictions

oos_results["second_score"] = second_scores

oos_results["prediction_margin"] = prediction_margins


# ------------------------------------------------------------
# 4. Display individual predictions
# ------------------------------------------------------------

print("\n===== OUT-OF-SCOPE PREDICTIONS =====")

display(
    oos_results[
        [
            "example_id",
            "out_of_scope_type",
            "predicted_category",
            "top_score",
            "second_prediction",
            "second_score",
            "prediction_margin",
        ]
    ].round(3)
)


# ------------------------------------------------------------
# 5. Summary statistics
# ------------------------------------------------------------

oos_summary = pd.DataFrame([{
    "test_cases": len(oos_results),
    "unique_predicted_categories": (
        oos_results["predicted_category"].nunique()
    ),
    "average_top_score": (
        oos_results["top_score"].mean()
    ),
    "minimum_top_score": (
        oos_results["top_score"].min()
    ),
    "maximum_top_score": (
        oos_results["top_score"].max()
    ),
    "average_prediction_margin": (
        oos_results["prediction_margin"].mean()
    ),
    "highest_top_score": (
        oos_results["top_score"].max()
    ),
}])

print("\n===== OUT-OF-SCOPE SUMMARY =====")

display(oos_summary.round(4))


# ------------------------------------------------------------
# 6. Predicted category distribution
# ------------------------------------------------------------

print("\n===== PREDICTED CATEGORY DISTRIBUTION =====")

display(
    oos_results["predicted_category"]
    .value_counts()
    .rename_axis("predicted_category")
    .to_frame("count")
)

OUT-OF-SCOPE TEST CASES: 10

===== OUT-OF-SCOPE PREDICTIONS =====


,example_id,out_of_scope_type,predicted_category,top_score,second_prediction,second_score,prediction_margin
55,OOS-001,Weather inquiry,REFUND,0.176,ACCOUNT,0.153,0.023
56,OOS-002,General knowledge,FEEDBACK,0.170,DELIVERY,0.163,0.007
57,OOS-003,Cooking request,FEEDBACK,0.282,REFUND,0.259,0.023
58,OOS-004,Sports inquiry,CANCEL,0.435,ACCOUNT,0.161,0.274
59,OOS-005,Translation request,CONTACT,0.156,REFUND,0.143,0.013
60,OOS-006,Creative writing,FEEDBACK,0.436,ACCOUNT,0.204,0.232
61,OOS-007,General knowledge,DELIVERY,0.247,CANCEL,0.153,0.094
62,OOS-008,Mathematical request,ORDER,0.256,ACCOUNT,0.211,0.045
63,OOS-009,Recommendation request,FEEDBACK,0.429,ACCOUNT,0.217,0.212
64,OOS-010,Programming knowledge,CANCEL,0.335,ACCOUNT,0.210,0.125



===== OUT-OF-SCOPE SUMMARY =====


,test_cases,unique_predicted_categories,average_top_score,minimum_top_score,maximum_top_score,average_prediction_margin,highest_top_score
0,10,6,0.2921,0.1557,0.4361,0.1047,0.4361



===== PREDICTED CATEGORY DISTRIBUTION =====


,count
predicted_category,
FEEDBACK,4
CANCEL,2
REFUND,1
CONTACT,1
DELIVERY,1
ORDER,1


In [20]:
# Combine prediction scores from all diagnostic test groups

classification_scores = classification_results[
    ["example_id", "top_score", "prediction_margin"]
].copy()

classification_scores["test_group"] = "classification"
classification_scores["is_correct"] = (
    classification_results["predicted_category"]
    == classification_results["expected_category"]
).values

ambiguous_scores = ambiguous_results[
    ["example_id", "top_score", "prediction_margin"]
].copy()

ambiguous_scores["test_group"] = "ambiguous"
ambiguous_scores["is_correct"] = np.nan

oos_scores = oos_results[
    ["example_id", "top_score", "prediction_margin"]
].copy()

oos_scores["test_group"] = "out_of_scope"
oos_scores["is_correct"] = np.nan

# Combine all groups
score_analysis = pd.concat(
    [
        classification_scores,
        ambiguous_scores,
        oos_scores
    ],
    ignore_index=True
)

# Summary by test group
score_summary = (
    score_analysis
    .groupby("test_group")
    .agg(
        test_cases=("example_id", "count"),
        average_top_score=("top_score", "mean"),
        median_top_score=("top_score", "median"),
        minimum_top_score=("top_score", "min"),
        maximum_top_score=("top_score", "max"),
        average_margin=("prediction_margin", "mean"),
        median_margin=("prediction_margin", "median")
    )
    .round(4)
)

print("===== SCORE COMPARISON =====")
display(score_summary)

# Compare correct and incorrect classification predictions
classification_reliability = (
    score_analysis[
        score_analysis["test_group"] == "classification"
    ]
    .groupby("is_correct")
    .agg(
        test_cases=("example_id", "count"),
        average_top_score=("top_score", "mean"),
        median_top_score=("top_score", "median"),
        average_margin=("prediction_margin", "mean"),
        median_margin=("prediction_margin", "median")
    )
    .round(4)
)

print("\n===== CLASSIFICATION: CORRECT VS INCORRECT =====")
display(classification_reliability)

===== SCORE COMPARISON =====


,test_cases,average_top_score,median_top_score,minimum_top_score,maximum_top_score,average_margin,median_margin
test_group,,,,,,,
ambiguous,11,0.4322,0.4402,0.2050,0.6722,0.2453,0.2368
classification,44,0.5204,0.4652,0.1691,0.9821,0.3761,0.2220
out_of_scope,10,0.2921,0.2691,0.1557,0.4361,0.1047,0.0692



===== CLASSIFICATION: CORRECT VS INCORRECT =====


,test_cases,average_top_score,median_top_score,average_margin,median_margin
is_correct,,,,,
False,17,0.3067,0.2605,0.1202,0.0629
True,27,0.6549,0.6368,0.5373,0.5674


In [21]:
# 1. Correct predictions with the lowest scores
low_confidence_correct = (
    classification_results[
        classification_results["predicted_category"]
        == classification_results["expected_category"]
    ]
    .sort_values("top_score")
    .head(5)
)

# 2. Incorrect predictions with the highest scores
high_confidence_errors = (
    classification_results[
        classification_results["predicted_category"]
        != classification_results["expected_category"]
    ]
    .sort_values("top_score", ascending=False)
    .head(5)
)

print("===== LOW-CONFIDENCE CORRECT PREDICTIONS =====")

display(
    low_confidence_correct[
        [
            "example_id",
            "text",
            "expected_category",
            "predicted_category",
            "top_score",
            "prediction_margin"
        ]
    ]
)

print("\n===== HIGH-CONFIDENCE INCORRECT PREDICTIONS =====")

display(
    high_confidence_errors[
        [
            "example_id",
            "text",
            "expected_category",
            "predicted_category",
            "top_score",
            "prediction_margin"
        ]
    ]
)

# Save the combined score analysis
score_analysis.to_csv(
    "data/benchmark/score_analysis_v1.csv",
    index=False
)

score_summary.to_csv(
    "data/benchmark/score_summary_v1.csv"
)

classification_reliability.to_csv(
    "data/benchmark/classification_reliability_v1.csv"
)

print("\nScore analysis files saved successfully.")

===== LOW-CONFIDENCE CORRECT PREDICTIONS =====


,example_id,text,expected_category,predicted_category,top_score,prediction_margin
43,PT-044,Where can I manage the emails I am subscribed to?,SUBSCRIPTION,SUBSCRIPTION,0.169089,0.009310
13,PT-014,I need an update on when the courier will brin...,DELIVERY,DELIVERY,0.263549,0.057912
24,PT-025,I want to know whether my purchase has been co...,ORDER,ORDER,0.280754,0.105491
27,PT-028,Where can I find the reference number for my r...,ORDER,ORDER,0.312536,0.142692
25,PT-026,Can I review the details of the order I placed...,ORDER,ORDER,0.357284,0.133636



===== HIGH-CONFIDENCE INCORRECT PREDICTIONS =====


,example_id,text,expected_category,predicted_category,top_score,prediction_margin
7,PT-008,I would like to cancel the order I placed earl...,CANCEL,ORDER,0.621440,0.527017
18,PT-019,"After completing my purchase, I noticed someth...",FEEDBACK,ORDER,0.535604,0.424788
16,PT-017,I would like to share my experience with your ...,FEEDBACK,CONTACT,0.512101,0.131626
10,PT-011,The automated replies have not resolved my iss...,CONTACT,ACCOUNT,0.390343,0.259199
42,PT-043,"The promotional emails arrive too frequently, ...",SUBSCRIPTION,DELIVERY,0.363299,0.049252


OSError: Cannot save file into a non-existent directory: 'data/benchmark'

In [22]:
from pathlib import Path

# Locate the Lumora project root
current_path = Path.cwd()

project_root = next(
    (
        path
        for path in [current_path, *current_path.parents]
        if (path / "data" / "benchmark").is_dir()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate Lumora project root."
    )

benchmark_dir = project_root / "data" / "benchmark"

# Save diagnostic results
score_analysis.to_csv(
    benchmark_dir / "score_analysis_v1.csv",
    index=False
)

score_summary.to_csv(
    benchmark_dir / "score_summary_v1.csv"
)

classification_reliability.to_csv(
    benchmark_dir / "classification_reliability_v1.csv"
)

print("Project root:", project_root)
print("Benchmark directory:", benchmark_dir)
print("All score analysis files saved successfully.")

Project root: /Users/raghad/Desktop/Lumora
Benchmark directory: /Users/raghad/Desktop/Lumora/data/benchmark
All score analysis files saved successfully.


In [23]:
# Identify incorrect predictions
error_analysis = classification_results[
    classification_results["predicted_category"]
    != classification_results["expected_category"]
].copy()

# Create confusion pairs
error_analysis["confusion_pair"] = (
    error_analysis["expected_category"]
    + " → "
    + error_analysis["predicted_category"]
)

# Summarize repeated confusion patterns
error_patterns = (
    error_analysis
    .groupby("confusion_pair")
    .agg(
        error_count=("example_id", "count"),
        average_top_score=("top_score", "mean"),
        average_margin=("prediction_margin", "mean"),
        example_ids=("example_id", lambda x: ", ".join(x))
    )
    .sort_values(
        ["error_count", "average_top_score"],
        ascending=[False, False]
    )
    .round(4)
    .reset_index()
)

print("===== ERROR PATTERN SUMMARY =====")
display(error_patterns)

print("\n===== ALL MISCLASSIFIED EXAMPLES =====")

display(
    error_analysis[
        [
            "example_id",
            "text",
            "expected_category",
            "predicted_category",
            "top_score",
            "prediction_margin",
            "confusion_pair"
        ]
    ].sort_values("confusion_pair")
)

# Save results
error_analysis.to_csv(
    benchmark_dir / "classification_errors_v1.csv",
    index=False
)

error_patterns.to_csv(
    benchmark_dir / "error_patterns_v1.csv",
    index=False
)

print("\nError analysis files saved successfully.")

===== ERROR PATTERN SUMMARY =====


,confusion_pair,error_count,average_top_score,average_margin,example_ids
0,PAYMENT → ACCOUNT,3,0.2214,0.0790,"PT-030, PT-031, PT-032"
1,CANCEL → ACCOUNT,2,0.1901,0.0335,"PT-005, PT-007"
2,CANCEL → ORDER,1,0.6214,0.5270,PT-008
3,FEEDBACK → ORDER,1,0.5356,0.4248,PT-019
4,FEEDBACK → CONTACT,1,0.5121,0.1316,PT-017
5,CONTACT → ACCOUNT,1,0.3903,0.2592,PT-011
6,SUBSCRIPTION → DELIVERY,1,0.3633,0.0493,PT-043
7,CANCEL → CONTACT,1,0.2924,0.0603,PT-006
8,ORDER → ACCOUNT,1,0.2731,0.1098,PT-027
9,DELIVERY → ACCOUNT,1,0.2659,0.0369,PT-013



===== ALL MISCLASSIFIED EXAMPLES =====


,example_id,text,expected_category,predicted_category,top_score,prediction_margin,confusion_pair
4,PT-005,Please terminate my current service before the...,CANCEL,ACCOUNT,0.190964,0.024726,CANCEL → ACCOUNT
6,PT-007,"My renewal is approaching, so I want to stop t...",CANCEL,ACCOUNT,0.189318,0.042279,CANCEL → ACCOUNT
5,PT-006,I no longer wish to continue using this service.,CANCEL,CONTACT,0.292411,0.060313,CANCEL → CONTACT
7,PT-008,I would like to cancel the order I placed earl...,CANCEL,ORDER,0.621440,0.527017,CANCEL → ORDER
10,PT-011,The automated replies have not resolved my iss...,CONTACT,ACCOUNT,0.390343,0.259199,CONTACT → ACCOUNT
12,PT-013,Can you tell me the estimated arrival date of ...,DELIVERY,ACCOUNT,0.265854,0.036926,DELIVERY → ACCOUNT
14,PT-015,The tracking page has not changed since yester...,DELIVERY,REFUND,0.195909,0.009454,DELIVERY → REFUND
16,PT-017,I would like to share my experience with your ...,FEEDBACK,CONTACT,0.512101,0.131626,FEEDBACK → CONTACT
18,PT-019,"After completing my purchase, I noticed someth...",FEEDBACK,ORDER,0.535604,0.424788,FEEDBACK → ORDER
26,PT-027,"I completed checkout, but no confirmation appe...",ORDER,ACCOUNT,0.273076,0.109832,ORDER → ACCOUNT



Error analysis files saved successfully.


In [24]:
# Inspect original training examples for confusing categories

training_data = datasets["training"].copy()

focus_categories = [
    "PAYMENT",
    "CANCEL",
    "ACCOUNT",
    "FEEDBACK",
    "CONTACT",
    "ORDER",
    "DELIVERY",
    "SHIPPING",
    "SUBSCRIPTION",
    "REFUND"
]

# 1. Count training examples per category and intent
intent_distribution = (
    training_data[
        training_data["category"].isin(focus_categories)
    ]
    .groupby(["category", "intent"])
    .size()
    .reset_index(name="example_count")
    .sort_values(
        ["category", "example_count"],
        ascending=[True, False]
    )
)

print("===== TRAINING INTENT DISTRIBUTION =====")
display(intent_distribution)

# 2. Select representative examples from each intent
training_samples = (
    training_data[
        training_data["category"].isin(focus_categories)
    ]
    .groupby(["category", "intent"], group_keys=False)
    .apply(
        lambda group: group.sample(
            n=min(3, len(group)),
            random_state=42
        ),
        include_groups=True
    )
    [["category", "intent", "instruction"]]
    .sort_values(["category", "intent"])
    .reset_index(drop=True)
)

print("\n===== REPRESENTATIVE TRAINING EXAMPLES =====")
display(training_samples)

print("\nTOTAL SAMPLED EXAMPLES:", len(training_samples))

===== TRAINING INTENT DISTRIBUTION =====


,category,intent,example_count
2,ACCOUNT,edit_account,1000
5,ACCOUNT,switch_account,1000
4,ACCOUNT,registration_problems,999
0,ACCOUNT,create_account,997
1,ACCOUNT,delete_account,995
3,ACCOUNT,recover_password,995
6,CANCEL,check_cancellation_fee,950
7,CONTACT,contact_customer_service,1000
8,CONTACT,contact_human_agent,999
10,DELIVERY,delivery_period,999


ValueError: include_groups=True is no longer allowed.

In [25]:
# Select representative training examples from each intent

training_samples = (
    training_data[
        training_data["category"].isin(focus_categories)
    ]
    .groupby(["category", "intent"], group_keys=False)
    .sample(frac=1, random_state=42)
    .groupby(["category", "intent"], group_keys=False)
    .head(3)
    [["category", "intent", "instruction"]]
    .sort_values(["category", "intent"])
    .reset_index(drop=True)
)

print("===== REPRESENTATIVE TRAINING EXAMPLES =====")
display(training_samples)

print("\nTOTAL SAMPLED EXAMPLES:", len(training_samples))

===== REPRESENTATIVE TRAINING EXAMPLES =====


,category,intent,instruction
0,ACCOUNT,create_account,new {{Account Category}} acount for wife
1,ACCOUNT,create_account,help me creating a freemium account
2,ACCOUNT,create_account,i have a problem with opening a {{Account Cate...
3,ACCOUNT,delete_account,removing {{Account Category}} account
4,ACCOUNT,delete_account,I want help to close a standard account
...,...,...,...
70,SHIPPING,set_up_shipping_address,I do not know how to set a secondary shippung ...
71,SHIPPING,set_up_shipping_address,need to set my new shipping address up
72,SUBSCRIPTION,newsletter_subscription,wanna know about cancel the subscription to ur...
73,SUBSCRIPTION,newsletter_subscription,tell me about cancellling the subscription to ...



TOTAL SAMPLED EXAMPLES: 75


In [26]:
# Display all relevant training examples without truncation

import pandas as pd

error_related_categories = [
    "PAYMENT",
    "CANCEL",
    "ACCOUNT",
    "FEEDBACK",
    "CONTACT",
    "SUBSCRIPTION"
]

relevant_samples = (
    training_samples[
        training_samples["category"].isin(error_related_categories)
    ]
    .sort_values(["category", "intent"])
    .reset_index(drop=True)
)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_colwidth", None,
    "display.width", 200
):
    display(relevant_samples)

print("Total relevant examples:", len(relevant_samples))

,category,intent,instruction
0,ACCOUNT,create_account,new {{Account Category}} acount for wife
1,ACCOUNT,create_account,help me creating a freemium account
2,ACCOUNT,create_account,i have a problem with opening a {{Account Category}} account
3,ACCOUNT,delete_account,removing {{Account Category}} account
4,ACCOUNT,delete_account,I want help to close a standard account
5,ACCOUNT,delete_account,how can i get information about my gold account removal
6,ACCOUNT,edit_account,I'm trying to correct the info included on my profile
7,ACCOUNT,edit_account,edit details on platinum account
8,ACCOUNT,edit_account,"I have got to edit my damn user, could I get some help?"
9,ACCOUNT,recover_password,I am trying to restore my user profile key


Total relevant examples: 42


In [27]:
# Display all classification errors with their prediction scores

error_columns = [
    "example_id",
    "text",
    "expected_category",
    "predicted_category",
    "top_probability",
    "margin"
]

with pd.option_context(
    "display.max_rows", 100,
    "display.max_colwidth", None,
    "display.width", 220
):
    display(
        classification_errors[error_columns]
        .sort_values(["expected_category", "example_id"])
        .reset_index(drop=True)
    )

print("Total classification errors:", len(classification_errors))

NameError: name 'classification_errors' is not defined

In [28]:
print(classification_errors.columns.tolist())

NameError: name 'classification_errors' is not defined

In [29]:
from pathlib import Path
import pandas as pd

# Locate Lumora project root
current_path = Path.cwd()

project_root = next(
    (
        path for path in [current_path, *current_path.parents]
        if (path / "data" / "benchmark").is_dir()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError("Could not locate Lumora project root.")

benchmark_dir = project_root / "data" / "benchmark"

# Load classification error analysis
errors_file = benchmark_dir / "classification_errors_v1.csv"

if not errors_file.exists():
    raise FileNotFoundError(
        f"File not found: {errors_file}"
    )

classification_errors = pd.read_csv(errors_file)

print("Loaded:", errors_file.name)
print("Shape:", classification_errors.shape)
print("Columns:", classification_errors.columns.tolist())

Loaded: classification_errors_v1.csv
Shape: (17, 11)
Columns: ['example_id', 'text', 'expected_category', 'test_type', 'predicted_category', 'correct', 'top_score', 'second_prediction', 'second_score', 'prediction_margin', 'confusion_pair']


In [30]:
# Display all classification errors for detailed inspection

error_columns = [
    "example_id",
    "text",
    "expected_category",
    "predicted_category",
    "top_score",
    "second_prediction",
    "second_score",
    "prediction_margin",
    "test_type"
]

with pd.option_context(
    "display.max_rows", 100,
    "display.max_colwidth", None,
    "display.width", 250
):
    display(
        classification_errors[error_columns]
        .sort_values(["expected_category", "example_id"])
        .reset_index(drop=True)
    )

print("Total classification errors:", len(classification_errors))

,example_id,text,expected_category,predicted_category,top_score,second_prediction,second_score,prediction_margin,test_type
0,PT-005,Please terminate my current service before the next billing cycle.,CANCEL,ACCOUNT,0.190964,CANCEL,0.166238,0.024726,natural
1,PT-006,I no longer wish to continue using this service.,CANCEL,CONTACT,0.292411,ACCOUNT,0.232097,0.060313,paraphrase
2,PT-007,"My renewal is approaching, so I want to stop the service beforehand.",CANCEL,ACCOUNT,0.189318,REFUND,0.147039,0.042279,contextual
3,PT-008,I would like to cancel the order I placed earlier today.,CANCEL,ORDER,0.621440,ACCOUNT,0.094423,0.527017,natural
4,PT-011,"The automated replies have not resolved my issue, and I need human assistance.",CONTACT,ACCOUNT,0.390343,CONTACT,0.131144,0.259199,contextual
5,PT-013,Can you tell me the estimated arrival date of my package?,DELIVERY,ACCOUNT,0.265854,DELIVERY,0.228928,0.036926,natural
6,PT-015,"The tracking page has not changed since yesterday, and I am waiting for the parcel.",DELIVERY,REFUND,0.195909,ACCOUNT,0.186454,0.009454,contextual
7,PT-017,I would like to share my experience with your customer service.,FEEDBACK,CONTACT,0.512101,FEEDBACK,0.380475,0.131626,natural
8,PT-019,"After completing my purchase, I noticed something I would like to suggest improving.",FEEDBACK,ORDER,0.535604,ACCOUNT,0.110816,0.424788,contextual
9,PT-027,"I completed checkout, but no confirmation appeared on the screen.",ORDER,ACCOUNT,0.273076,REFUND,0.163244,0.109832,contextual


Total classification errors: 17


In [31]:
# Review potentially problematic test annotations

annotation_review_ids = [
    "PT-005",
    "PT-006",
    "PT-007",
    "PT-008",
    "PT-015",
    "PT-017",
    "PT-019",
    "PT-030",
    "PT-031",
    "PT-032",
    "PT-037",
    "PT-041",
    "PT-043"
]

annotation_review = (
    classification_test[
        classification_test["example_id"].isin(annotation_review_ids)
    ][
        [
            "example_id",
            "text",
            "expected_category",
            "test_type"
        ]
    ]
    .sort_values("example_id")
    .reset_index(drop=True)
)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_colwidth", None,
    "display.width", 220
):
    display(annotation_review)

,example_id,text,expected_category,test_type
0,PT-005,Please terminate my current service before the next billing cycle.,CANCEL,natural
1,PT-006,I no longer wish to continue using this service.,CANCEL,paraphrase
2,PT-007,"My renewal is approaching, so I want to stop the service beforehand.",CANCEL,contextual
3,PT-008,I would like to cancel the order I placed earlier today.,CANCEL,natural
4,PT-015,"The tracking page has not changed since yesterday, and I am waiting for the parcel.",DELIVERY,contextual
5,PT-017,I would like to share my experience with your customer service.,FEEDBACK,natural
6,PT-019,"After completing my purchase, I noticed something I would like to suggest improving.",FEEDBACK,contextual
7,PT-030,I need to replace the card used for future transactions.,PAYMENT,paraphrase
8,PT-031,The checkout page rejected my card even though it is still valid.,PAYMENT,contextual
9,PT-032,How can I verify whether my transaction was successfully charged?,PAYMENT,natural


In [32]:
# Build an intent-to-category reference table

intent_reference = (
    training_data
    .groupby(["category", "intent"])
    .agg(
        training_examples=("instruction", "size"),
        unique_instructions=("instruction", "nunique")
    )
    .reset_index()
    .sort_values(["category", "intent"])
)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_colwidth", None,
    "display.width", 180
):
    display(intent_reference)

print("Total categories:", intent_reference["category"].nunique())
print("Total intents:", intent_reference["intent"].nunique())

,category,intent,training_examples,unique_instructions
0,ACCOUNT,create_account,997,892
1,ACCOUNT,delete_account,995,918
2,ACCOUNT,edit_account,1000,777
3,ACCOUNT,recover_password,995,995
4,ACCOUNT,registration_problems,999,999
5,ACCOUNT,switch_account,1000,862
6,CANCEL,check_cancellation_fee,950,950
7,CONTACT,contact_customer_service,1000,1000
8,CONTACT,contact_human_agent,999,999
9,DELIVERY,delivery_options,995,660


Total categories: 11
Total intents: 27


In [33]:
# Inspect training examples for the most relevant intents

intents_to_inspect = [
    "check_cancellation_fee",
    "cancel_order",
    "track_order",
    "delivery_period",
    "payment_issue"
]

intent_samples = []

for intent_name in intents_to_inspect:

    group = training_data[
        training_data["intent"] == intent_name
    ]

    sample = group.sample(
        n=min(8, len(group)),
        random_state=42
    )

    intent_samples.append(sample)

intent_samples_df = (
    pd.concat(intent_samples, ignore_index=True)
    [["category", "intent", "instruction"]]
    .sort_values(["category", "intent"])
    .reset_index(drop=True)
)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_colwidth", None,
    "display.width", 220
):
    display(intent_samples_df)

print("Total inspected examples:", len(intent_samples_df))

,category,intent,instruction
0,CANCEL,check_cancellation_fee,I am tring to see the early exit penalty
1,CANCEL,check_cancellation_fee,check cancellation fee
2,CANCEL,check_cancellation_fee,I do not know how to check the cancellation charge
3,CANCEL,check_cancellation_fee,I want help to see the early exit fees
4,CANCEL,check_cancellation_fee,checking early exit fee
5,CANCEL,check_cancellation_fee,i have to check the cancellation penalty
6,CANCEL,check_cancellation_fee,I don't know what I have to do to see the termination fees
7,CANCEL,check_cancellation_fee,I have got to check the early exit fee
8,DELIVERY,delivery_period,I need assistance checking how soon can I expect my delivery
9,DELIVERY,delivery_period,help to check how soon can I expect my shipment


Total inspected examples: 40


In [34]:
# Inspect the remaining relevant training intents

remaining_intents = [
    "check_payment_methods",
    "delivery_options"
]

remaining_samples = []

for intent_name in remaining_intents:

    group = training_data[
        training_data["intent"] == intent_name
    ]

    sample = group.sample(
        n=min(10, len(group)),
        random_state=42
    )

    remaining_samples.append(sample)

remaining_samples_df = (
    pd.concat(remaining_samples, ignore_index=True)
    [["category", "intent", "instruction"]]
    .sort_values(["category", "intent"])
    .reset_index(drop=True)
)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_colwidth", None,
    "display.width", 220
):
    display(remaining_samples_df)

print("Total inspected examples:", len(remaining_samples_df))

,category,intent,instruction
0,DELIVERY,delivery_options,can I place an ordewr from {{Delivery Country}}?
1,DELIVERY,delivery_options,help to check what methods for delivery i have
2,DELIVERY,delivery_options,do you shp to {{Delivery City}}?
3,DELIVERY,delivery_options,is it possible to place an order from {{Delivery City}}?
4,DELIVERY,delivery_options,I am trying to check what delivery methods you offer
5,DELIVERY,delivery_options,wanna see what delivery options are there i need help
6,DELIVERY,delivery_options,"I need to see what delivery methods you offer, help me"
7,DELIVERY,delivery_options,show me what shipping options you offer
8,DELIVERY,delivery_options,I want information about the delivery methods
9,DELIVERY,delivery_options,I have got to check what delivery methods are there


Total inspected examples: 20


In [35]:
# Create an annotation audit log

annotation_audit = pd.DataFrame([
    {
        "example_id": "PT-008",
        "original_category": "CANCEL",
        "proposed_category": "ORDER",
        "audit_status": "proposed_relabel",
        "reason": "The request explicitly asks to cancel a purchase order, matching ORDER -> cancel_order."
    },
    {
        "example_id": "PT-015",
        "original_category": "DELIVERY",
        "proposed_category": "ORDER",
        "audit_status": "proposed_relabel",
        "reason": "The request concerns an unchanged tracking page, matching ORDER -> track_order more closely."
    },
    {
        "example_id": "PT-005",
        "original_category": "CANCEL",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": "Service termination is not clearly represented by the available CANCEL training intent."
    },
    {
        "example_id": "PT-006",
        "original_category": "CANCEL",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": "Service discontinuation is not clearly represented by the available CANCEL training intent."
    },
    {
        "example_id": "PT-007",
        "original_category": "CANCEL",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": "Stopping a service before renewal is not clearly represented by the available CANCEL training intent."
    },
    {
        "example_id": "PT-030",
        "original_category": "PAYMENT",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": "Replacing a payment card requires checking the intended scope of PAYMENT."
    }
])

display(annotation_audit)

,example_id,original_category,proposed_category,audit_status,reason
0,PT-008,CANCEL,ORDER,proposed_relabel,The request explicitly asks to cancel a purcha...
1,PT-015,DELIVERY,ORDER,proposed_relabel,The request concerns an unchanged tracking pag...
2,PT-005,CANCEL,NaN,definition_review,Service termination is not clearly represented...
3,PT-006,CANCEL,NaN,definition_review,Service discontinuation is not clearly represe...
4,PT-007,CANCEL,NaN,definition_review,Stopping a service before renewal is not clear...
5,PT-030,PAYMENT,NaN,definition_review,Replacing a payment card requires checking the...


In [36]:
annotation_audit = pd.DataFrame([
    {
        "example_id": "PT-008",
        "original_category": "CANCEL",
        "proposed_category": "ORDER",
        "audit_status": "proposed_relabel",
        "reason": "The request explicitly asks to cancel a purchase order, matching ORDER -> cancel_order."
    },
    {
        "example_id": "PT-015",
        "original_category": "DELIVERY",
        "proposed_category": "ORDER",
        "audit_status": "proposed_relabel",
        "reason": "The request concerns an unchanged tracking page, matching ORDER -> track_order more closely."
    },
    {
        "example_id": "PT-005",
        "original_category": "CANCEL",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": "Service termination is not clearly represented by the available CANCEL training intent."
    },
    {
        "example_id": "PT-006",
        "original_category": "CANCEL",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": "Service discontinuation is not clearly represented by the available CANCEL training intent."
    },
    {
        "example_id": "PT-007",
        "original_category": "CANCEL",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": "Stopping a service before renewal is not clearly represented by the available CANCEL training intent."
    },
    {
        "example_id": "PT-030",
        "original_category": "PAYMENT",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": "Replacing a payment card requires checking the intended scope of PAYMENT."
    }
])

display(annotation_audit)

,example_id,original_category,proposed_category,audit_status,reason
0,PT-008,CANCEL,ORDER,proposed_relabel,The request explicitly asks to cancel a purcha...
1,PT-015,DELIVERY,ORDER,proposed_relabel,The request concerns an unchanged tracking pag...
2,PT-005,CANCEL,NaN,definition_review,Service termination is not clearly represented...
3,PT-006,CANCEL,NaN,definition_review,Service discontinuation is not clearly represe...
4,PT-007,CANCEL,NaN,definition_review,Stopping a service before renewal is not clear...
5,PT-030,PAYMENT,NaN,definition_review,Replacing a payment card requires checking the...


In [37]:
print("Columns:")
print(annotation_audit.columns.tolist())

print("\nFirst row:")
print(annotation_audit.iloc[0].to_dict())

print("\nFull DataFrame:")
display(annotation_audit)

Columns:
['example_id', 'original_category', 'proposed_category', 'audit_status', 'reason']

First row:
{'example_id': 'PT-008', 'original_category': 'CANCEL', 'proposed_category': 'ORDER', 'audit_status': 'proposed_relabel', 'reason': 'The request explicitly asks to cancel a purchase order, matching ORDER -> cancel_order.'}

Full DataFrame:


,example_id,original_category,proposed_category,audit_status,reason
0,PT-008,CANCEL,ORDER,proposed_relabel,The request explicitly asks to cancel a purcha...
1,PT-015,DELIVERY,ORDER,proposed_relabel,The request concerns an unchanged tracking pag...
2,PT-005,CANCEL,NaN,definition_review,Service termination is not clearly represented...
3,PT-006,CANCEL,NaN,definition_review,Service discontinuation is not clearly represe...
4,PT-007,CANCEL,NaN,definition_review,Stopping a service before renewal is not clear...
5,PT-030,PAYMENT,NaN,definition_review,Replacing a payment card requires checking the...


In [38]:
annotation_audit.to_csv(
    benchmark_dir / "annotation_audit_v1.csv",
    index=False
)

print("Annotation audit saved successfully.")

Annotation audit saved successfully.


In [39]:
remaining_audit = pd.DataFrame([
    {
        "example_id": "PT-011",
        "original_category": "CONTACT",
        "proposed_category": "CONTACT",
        "audit_status": "likely_model_error",
        "reason": "The request asks for human assistance after automated replies failed, which aligns with human-agent contact."
    },
    {
        "example_id": "PT-013",
        "original_category": "DELIVERY",
        "proposed_category": "DELIVERY",
        "audit_status": "likely_model_error",
        "reason": "The request asks for the estimated package arrival date, matching the delivery_period intent."
    },
    {
        "example_id": "PT-017",
        "original_category": "FEEDBACK",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": "Sharing an experience with customer service may overlap with CONTACT and FEEDBACK."
    },
    {
        "example_id": "PT-019",
        "original_category": "FEEDBACK",
        "proposed_category": "FEEDBACK",
        "audit_status": "likely_model_error",
        "reason": "The request suggests an improvement, which aligns with feedback despite mentioning a purchase."
    },
    {
        "example_id": "PT-027",
        "original_category": "ORDER",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": "Missing order confirmation after checkout may relate to order status or a system/account issue."
    },
    {
        "example_id": "PT-031",
        "original_category": "PAYMENT",
        "proposed_category": "PAYMENT",
        "audit_status": "likely_model_error",
        "reason": "A card being rejected during checkout is directly related to a payment issue."
    },
    {
        "example_id": "PT-032",
        "original_category": "PAYMENT",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": "Verifying whether a transaction was charged may require a dedicated transaction-status intent."
    },
    {
        "example_id": "PT-037",
        "original_category": "SHIPPING",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": "Requesting delivery to a different destination may concern shipping-address changes; taxonomy scope needs confirmation."
    },
    {
        "example_id": "PT-039",
        "original_category": "SHIPPING",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": "The request may concern updating a shipping address; confirm the exact intent scope."
    },
    {
        "example_id": "PT-041",
        "original_category": "SUBSCRIPTION",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": "Managing email update preferences may extend beyond the available newsletter_subscription examples."
    },
    {
        "example_id": "PT-043",
        "original_category": "SUBSCRIPTION",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": "Managing promotional email frequency may not be sufficiently represented by the newsletter_subscription intent."
    }
])

annotation_audit = pd.concat(
    [annotation_audit, remaining_audit],
    ignore_index=True
)

# Validate the audit log
assert annotation_audit["example_id"].is_unique
assert len(annotation_audit) == 17

# Save the complete audit log
annotation_audit.to_csv(
    benchmark_dir / "annotation_audit_v1.csv",
    index=False
)

display(annotation_audit)

print("\nAudit status summary:")
display(annotation_audit["audit_status"].value_counts())

print("\nTotal audited cases:", len(annotation_audit))
print("Saved successfully.")

,example_id,original_category,proposed_category,audit_status,reason
0,PT-008,CANCEL,ORDER,proposed_relabel,The request explicitly asks to cancel a purcha...
1,PT-015,DELIVERY,ORDER,proposed_relabel,The request concerns an unchanged tracking pag...
2,PT-005,CANCEL,NaN,definition_review,Service termination is not clearly represented...
3,PT-006,CANCEL,NaN,definition_review,Service discontinuation is not clearly represe...
4,PT-007,CANCEL,NaN,definition_review,Stopping a service before renewal is not clear...
5,PT-030,PAYMENT,NaN,definition_review,Replacing a payment card requires checking the...
6,PT-011,CONTACT,CONTACT,likely_model_error,The request asks for human assistance after au...
7,PT-013,DELIVERY,DELIVERY,likely_model_error,The request asks for the estimated package arr...
8,PT-017,FEEDBACK,NaN,definition_review,Sharing an experience with customer service ma...
9,PT-019,FEEDBACK,FEEDBACK,likely_model_error,"The request suggests an improvement, which ali..."



Audit status summary:


audit_status
definition_review     11
likely_model_error     4
proposed_relabel       2
Name: count, dtype: int64


Total audited cases: 17
Saved successfully.


In [40]:
from pathlib import Path
import pandas as pd

# Locate Lumora project
current_path = Path.cwd()

project_root = next(
    (
        path for path in [current_path, *current_path.parents]
        if (path / "data" / "raw").is_dir()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError("Could not locate Lumora project root.")

# Load original training dataset
dataset_path = project_root / "data" / "raw" / "customer_support_tickets.csv"
training_data = pd.read_csv(dataset_path)

# Filter SHIPPING examples
shipping_data = training_data[
    training_data["category"] == "SHIPPING"
].copy()

# Display available SHIPPING intents
print("SHIPPING intents and training counts:")
display(
    shipping_data["intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="training_examples")
)

# Inspect representative examples from each intent
for intent in sorted(shipping_data["intent"].unique()):

    intent_examples = shipping_data[
        shipping_data["intent"] == intent
    ]

    print(f"\n{'=' * 60}")
    print(f"Intent: {intent}")
    print(f"Total examples: {len(intent_examples)}")
    print("=" * 60)

    display(
        intent_examples[
            ["category", "intent", "instruction"]
        ].sample(
            n=min(10, len(intent_examples)),
            random_state=42
        ).reset_index(drop=True)
    )

SHIPPING intents and training counts:


,intent,training_examples
0,set_up_shipping_address,997
1,change_shipping_address,973



Intent: change_shipping_address
Total examples: 973


,category,intent,instruction
0,SHIPPING,change_shipping_address,there is an issue trying to update the address
1,SHIPPING,change_shipping_address,I cannot edit my delivery address
2,SHIPPING,change_shipping_address,I want assistance updating my delivery address
3,SHIPPING,change_shipping_address,i want information about correcting the shipin...
4,SHIPPING,change_shipping_address,"I entered a wrong delivery address, help me ed..."
5,SHIPPING,change_shipping_address,I have a trouble correcing the address
6,SHIPPING,change_shipping_address,I have an error updating my address
7,SHIPPING,change_shipping_address,problems with a delivery address modification
8,SHIPPING,change_shipping_address,"I want to modify the address, how can I do it?"
9,SHIPPING,change_shipping_address,"I have entered a wrong delivery address, help ..."



Intent: set_up_shipping_address
Total examples: 997


,category,intent,instruction
0,SHIPPING,set_up_shipping_address,i want help submitting a different delivery ad...
1,SHIPPING,set_up_shipping_address,error submitting my shipping address
2,SHIPPING,set_up_shipping_address,there is a trouble setting another delivery ad...
3,SHIPPING,set_up_shipping_address,there is an issue submitting the new delivery ...
4,SHIPPING,set_up_shipping_address,"I have to enter the shipping address, how do I..."
5,SHIPPING,set_up_shipping_address,I need to submit my delivery address
6,SHIPPING,set_up_shipping_address,"I have got to submit the new shipping address,..."
7,SHIPPING,set_up_shipping_address,could you help me setting up my new delivery a...
8,SHIPPING,set_up_shipping_address,I need assistance to set the delivery address up
9,SHIPPING,set_up_shipping_address,can you help me setting another delivery addrs...


In [41]:
# Update shipping audit decisions

shipping_updates = {
    "PT-037": {
        "proposed_category": "SHIPPING",
        "audit_status": "likely_model_error",
        "reason": (
            "The request asks to send a package to a different destination, "
            "which aligns with SHIPPING address-change intents."
        )
    },
    "PT-039": {
        "proposed_category": "SHIPPING",
        "audit_status": "likely_model_error",
        "reason": (
            "The request concerns updating a shipping address, "
            "which aligns with the change_shipping_address intent."
        )
    }
}

for example_id, update in shipping_updates.items():
    mask = annotation_audit["example_id"] == example_id

    for column, value in update.items():
        annotation_audit.loc[mask, column] = value

# Save updated audit
annotation_audit.to_csv(
    benchmark_dir / "annotation_audit_v1.csv",
    index=False
)

display(
    annotation_audit[
        annotation_audit["example_id"].isin(["PT-037", "PT-039"])
    ]
)

print("Shipping audit updated and saved.")

,example_id,original_category,proposed_category,audit_status,reason
13,PT-037,SHIPPING,SHIPPING,likely_model_error,The request asks to send a package to a differ...
14,PT-039,SHIPPING,SHIPPING,likely_model_error,The request concerns updating a shipping addre...


Shipping audit updated and saved.


In [42]:
# Inspect SUBSCRIPTION training examples

subscription_data = training_data[
    training_data["category"] == "SUBSCRIPTION"
].copy()

print("SUBSCRIPTION intents and counts:")

display(
    subscription_data["intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="training_examples")
)

for intent in sorted(subscription_data["intent"].unique()):

    intent_examples = subscription_data[
        subscription_data["intent"] == intent
    ]

    print(f"\n{'=' * 60}")
    print(f"Intent: {intent}")
    print(f"Total examples: {len(intent_examples)}")
    print("=" * 60)

    display(
        intent_examples[["category", "intent", "instruction"]]
        .sample(n=min(15, len(intent_examples)), random_state=42)
        .reset_index(drop=True)
    )

SUBSCRIPTION intents and counts:


,intent,training_examples
0,newsletter_subscription,999



Intent: newsletter_subscription
Total examples: 999


,category,intent,instruction
0,SUBSCRIPTION,newsletter_subscription,i want assistance unsubscribing from the newsl...
1,SUBSCRIPTION,newsletter_subscription,"I need to unsubscribe from the newsletter, can..."
2,SUBSCRIPTION,newsletter_subscription,can uhelp me subscribe to ur newsletter
3,SUBSCRIPTION,newsletter_subscription,I acn't unsubscribe to your company newsletter
4,SUBSCRIPTION,newsletter_subscription,I want help receoving the newsletter
5,SUBSCRIPTION,newsletter_subscription,information about receiving your company newsl...
6,SUBSCRIPTION,newsletter_subscription,"I have to receive your corporate newsletter, h..."
7,SUBSCRIPTION,newsletter_subscription,can I unsubscribe to your corporate newsletter?
8,SUBSCRIPTION,newsletter_subscription,I need to receive thecorporate newsletter
9,SUBSCRIPTION,newsletter_subscription,I can't receive the company newsletter


In [43]:
# Add subscription cases to the annotation audit

from pathlib import Path
import pandas as pd

audit_path = benchmark_dir / "annotation_audit_v1.csv"

annotation_audit = pd.read_csv(audit_path)

subscription_updates = pd.DataFrame([
    {
        "example_id": "PT-041",
        "original_category": "SUBSCRIPTION",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": (
            "Managing email update frequency may extend beyond the "
            "current newsletter_subscription intent, which mainly "
            "covers subscribing, unsubscribing, and subscription status."
        )
    },
    {
        "example_id": "PT-043",
        "original_category": "SUBSCRIPTION",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": (
            "Adjusting promotional email frequency and preferences "
            "is not explicitly represented in the sampled "
            "newsletter_subscription training examples."
        )
    }
])

# Remove existing versions of these IDs before updating
annotation_audit = annotation_audit[
    ~annotation_audit["example_id"].isin(
        subscription_updates["example_id"]
    )
]

annotation_audit = pd.concat(
    [annotation_audit, subscription_updates],
    ignore_index=True
)

annotation_audit = annotation_audit.sort_values(
    "example_id"
).reset_index(drop=True)

annotation_audit.to_csv(audit_path, index=False)

print("AUDIT UPDATED")
print("Total audited cases:", len(annotation_audit))
print("\nAudit status summary:")
display(annotation_audit["audit_status"].value_counts())

print("\nSubscription cases:")
display(
    annotation_audit[
        annotation_audit["example_id"].isin(["PT-041", "PT-043"])
    ]
)

AUDIT UPDATED
Total audited cases: 17

Audit status summary:


audit_status
definition_review     9
likely_model_error    6
proposed_relabel      2
Name: count, dtype: int64


Subscription cases:


,example_id,original_category,proposed_category,audit_status,reason
15,PT-041,SUBSCRIPTION,None,definition_review,Managing email update frequency may extend bey...
16,PT-043,SUBSCRIPTION,None,definition_review,Adjusting promotional email frequency and pref...


In [44]:
import pandas as pd

# Locate project files
project_root = benchmark_dir.parent

# Load original training data and test suite
training_data = pd.read_csv(
    project_root / "data" / "raw" / "customer_support_tickets.csv"
)

test_suite = pd.read_csv(
    benchmark_dir / "predeployment_test_v1.csv"
)

# 1. Inspect CANCEL training examples
cancel_samples = (
    training_data[
        (training_data["category"] == "CANCEL") &
        (training_data["intent"] == "check_cancellation_fee")
    ]["instruction"]
    .drop_duplicates()
    .head(10)
)

print("=" * 80)
print("CANCEL — check_cancellation_fee")
print("=" * 80)

for i, text in enumerate(cancel_samples, 1):
    print(f"{i}. {text}\n")


# 2. Inspect ACCOUNT deletion examples
delete_samples = (
    training_data[
        (training_data["category"] == "ACCOUNT") &
        (training_data["intent"] == "delete_account")
    ]["instruction"]
    .drop_duplicates()
    .head(10)
)

print("=" * 80)
print("ACCOUNT — delete_account")
print("=" * 80)

for i, text in enumerate(delete_samples, 1):
    print(f"{i}. {text}\n")


# 3. Inspect the three disputed test cases
cancel_cases = (
    test_suite[
        test_suite["example_id"].isin(["PT-005", "PT-006", "PT-007"])
    ][["example_id", "text", "expected_category", "annotation_rationale"]]
    .sort_values("example_id")
)

print("=" * 80)
print("DISPUTED TEST CASES")
print("=" * 80)

display(cancel_cases)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/raghad/Desktop/Lumora/data/data/raw/customer_support_tickets.csv'

In [45]:
import pandas as pd

# Project root
project_root = benchmark_dir.parent.parent

# Load original training data
training_data = pd.read_csv(
    project_root / "data" / "raw" / "customer_support_tickets.csv"
)

# Load test suite
test_suite = pd.read_csv(
    benchmark_dir / "predeployment_test_v1.csv"
)

print("Project root:", project_root)
print("Training data shape:", training_data.shape)


# 1. CANCEL training examples
cancel_samples = (
    training_data[
        (training_data["category"] == "CANCEL") &
        (training_data["intent"] == "check_cancellation_fee")
    ]["instruction"]
    .drop_duplicates()
    .head(10)
)

print("\n" + "=" * 80)
print("CANCEL — check_cancellation_fee")
print("=" * 80)

for i, text in enumerate(cancel_samples, 1):
    print(f"{i}. {text}\n")


# 2. ACCOUNT deletion examples
delete_samples = (
    training_data[
        (training_data["category"] == "ACCOUNT") &
        (training_data["intent"] == "delete_account")
    ]["instruction"]
    .drop_duplicates()
    .head(10)
)

print("\n" + "=" * 80)
print("ACCOUNT — delete_account")
print("=" * 80)

for i, text in enumerate(delete_samples, 1):
    print(f"{i}. {text}\n")


# 3. Disputed test cases
cancel_cases = (
    test_suite[
        test_suite["example_id"].isin(
            ["PT-005", "PT-006", "PT-007"]
        )
    ][
        ["example_id", "text", "expected_category", "annotation_rationale"]
    ]
    .sort_values("example_id")
)

print("\n" + "=" * 80)
print("DISPUTED TEST CASES")
print("=" * 80)

display(cancel_cases)

Project root: /Users/raghad/Desktop/Lumora
Training data shape: (26872, 5)

CANCEL — check_cancellation_fee
1. I can't ifnd the bloody termination charge, I need help

2. what is the fee for canceling the contract?

3. could I check the withdrawal charge?

4. i need help to see the early termination penalty

5. need assistance to see the withdrawal fee

6. check the withdrawal fees

7. can ushow me the withdrawal penalties

8. where can I see the early termination penalty?

9. I want to see the withdrawal penalties, I need assistance

10. wanna see the withdrawalcharge


ACCOUNT — delete_account
1. closing  platinum account

2. I don't know how I can remove my standard account

3. what do i have to do to remove the {{Account Type}} account

4. I want to delete a {{Account Type}} account, help me

5. I don't know what I have to do to remove my {{Account Category}} account

6. problem with freemium account cancellations

7. I do not know what I have to do to cancel a {{Account Category}}

,example_id,text,expected_category,annotation_rationale
4,PT-005,Please terminate my current service before the...,CANCEL,NaN
5,PT-006,I no longer wish to continue using this service.,CANCEL,NaN
6,PT-007,"My renewal is approaching, so I want to stop t...",CANCEL,NaN


In [46]:
# 1. All ACCOUNT intents and their counts
account_intents = (
    training_data[training_data["category"] == "ACCOUNT"]
    .groupby("intent")
    .size()
    .sort_values(ascending=False)
)

print("=" * 70)
print("ACCOUNT INTENTS")
print("=" * 70)
display(account_intents.to_frame("examples"))


# 2. ACCOUNT deletion examples
delete_samples = (
    training_data[
        (training_data["category"] == "ACCOUNT") &
        (training_data["intent"] == "delete_account")
    ]["instruction"]
    .drop_duplicates()
    .head(8)
)

print("\n" + "=" * 70)
print("ACCOUNT — delete_account")
print("=" * 70)

for i, text in enumerate(delete_samples, 1):
    print(f"{i}. {text}")


# 3. The three disputed cases without truncation
print("\n" + "=" * 70)
print("CANCEL DISPUTED CASES")
print("=" * 70)

for _, row in cancel_cases.iterrows():
    print(f"\n{row['example_id']}: {row['text']}")

ACCOUNT INTENTS


,examples
intent,
edit_account,1000
switch_account,1000
registration_problems,999
create_account,997
delete_account,995
recover_password,995



ACCOUNT — delete_account
1. closing  platinum account
2. I don't know how I can remove my standard account
3. what do i have to do to remove the {{Account Type}} account
4. I want to delete a {{Account Type}} account, help me
5. I don't know what I have to do to remove my {{Account Category}} account
6. problem with freemium account cancellations
7. I do not know what I have to do to cancel a {{Account Category}} account
8. delete {{Account Type}} account

CANCEL DISPUTED CASES

PT-005: Please terminate my current service before the next billing cycle.

PT-006: I no longer wish to continue using this service.

PT-007: My renewal is approaching, so I want to stop the service beforehand.


In [47]:
import pandas as pd

audit_path = benchmark_dir / "annotation_audit_v1.csv"
annotation_audit = pd.read_csv(audit_path)

cancel_updates = pd.DataFrame([
    {
        "example_id": "PT-005",
        "original_category": "CANCEL",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": (
            "The request concerns terminating an active service before "
            "the next billing cycle. The CANCEL training intent covers "
            "cancellation fees, while ACCOUNT/delete_account covers "
            "account removal. No explicit service-termination intent "
            "was identified."
        )
    },
    {
        "example_id": "PT-006",
        "original_category": "CANCEL",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": (
            "The request expresses a desire to discontinue a service "
            "without explicitly mentioning account deletion or "
            "cancellation fees. The current taxonomy does not clearly "
            "define where general service discontinuation belongs."
        )
    },
    {
        "example_id": "PT-007",
        "original_category": "CANCEL",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": (
            "The request concerns stopping a service before renewal. "
            "This may represent subscription or service cancellation, "
            "but the current taxonomy has no explicit intent for "
            "preventing service renewal."
        )
    }
])

annotation_audit = annotation_audit[
    ~annotation_audit["example_id"].isin(cancel_updates["example_id"])
]

annotation_audit = pd.concat(
    [annotation_audit, cancel_updates],
    ignore_index=True
)

annotation_audit = (
    annotation_audit
    .sort_values("example_id")
    .reset_index(drop=True)
)

annotation_audit.to_csv(audit_path, index=False)

print("CANCEL audit records updated successfully.")
print("\nAudit status summary:")
display(annotation_audit["audit_status"].value_counts())

print("\nUpdated CANCEL cases:")
display(
    annotation_audit[
        annotation_audit["example_id"].isin(
            ["PT-005", "PT-006", "PT-007"]
        )
    ]
)

CANCEL audit records updated successfully.

Audit status summary:


audit_status
definition_review     9
likely_model_error    6
proposed_relabel      2
Name: count, dtype: int64


Updated CANCEL cases:


,example_id,original_category,proposed_category,audit_status,reason
0,PT-005,CANCEL,None,definition_review,The request concerns terminating an active ser...
1,PT-006,CANCEL,None,definition_review,The request expresses a desire to discontinue ...
2,PT-007,CANCEL,None,definition_review,The request concerns stopping a service before...


In [48]:
# 1. Inspect PAYMENT intent distribution
payment_intents = (
    training_data[
        training_data["category"] == "PAYMENT"
    ]
    .groupby("intent")
    .size()
    .sort_values(ascending=False)
)

print("=" * 75)
print("PAYMENT INTENTS")
print("=" * 75)

display(payment_intents.to_frame("examples"))


# 2. Inspect check_payment_methods examples
methods_samples = (
    training_data[
        (training_data["category"] == "PAYMENT") &
        (training_data["intent"] == "check_payment_methods")
    ]["instruction"]
    .drop_duplicates()
    .head(8)
)

print("\n" + "=" * 75)
print("PAYMENT — check_payment_methods")
print("=" * 75)

for i, text in enumerate(methods_samples, 1):
    print(f"{i}. {text}")


# 3. Inspect payment_issue examples
issue_samples = (
    training_data[
        (training_data["category"] == "PAYMENT") &
        (training_data["intent"] == "payment_issue")
    ]["instruction"]
    .drop_duplicates()
    .head(8)
)

print("\n" + "=" * 75)
print("PAYMENT — payment_issue")
print("=" * 75)

for i, text in enumerate(issue_samples, 1):
    print(f"{i}. {text}")


# 4. Inspect disputed test cases
payment_cases = (
    test_suite[
        test_suite["example_id"].isin(
            ["PT-030", "PT-031", "PT-032"]
        )
    ][
        [
            "example_id",
            "text",
            "expected_category",
            "annotation_rationale"
        ]
    ]
    .sort_values("example_id")
)

print("\n" + "=" * 75)
print("PAYMENT DISPUTED CASES")
print("=" * 75)

display(payment_cases)

PAYMENT INTENTS


,examples
intent,
check_payment_methods,999
payment_issue,999



PAYMENT — check_payment_methods
1. I try to list your available payment methoids
2. I nneed help to list the accepted payment methods
3. could you help me check what payment options are available?
4. see ur payment methods
5. I have got to list the allowed payment modalities, help me
6. help me seeing your available payment options
7. I do not know how to check your allowed payment methods
8. need assikstance checking your allowed payment modalities

PAYMENT — payment_issue
1. i cant make a payment where can i notify of an error
2. problem with payments
3. can you help me notifying of errors with online payment?
4. I do not know how I can notify of a payment issue
5. can uhelpme informing of a problem with payments
6. where can I report problems with payments?
7. my card was declined help to inform oof a payment issue
8. I try to inform of a payment issue

PAYMENT DISPUTED CASES


,example_id,text,expected_category,annotation_rationale
29,PT-030,I need to replace the card used for future tra...,PAYMENT,NaN
30,PT-031,The checkout page rejected my card even though...,PAYMENT,NaN
31,PT-032,How can I verify whether my transaction was su...,PAYMENT,NaN


In [49]:
import pandas as pd

audit_path = benchmark_dir / "annotation_audit_v1.csv"
annotation_audit = pd.read_csv(audit_path)

payment_updates = pd.DataFrame([
    {
        "example_id": "PT-030",
        "original_category": "PAYMENT",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": (
            "The request concerns replacing a stored payment card "
            "for future transactions. Existing PAYMENT intents cover "
            "available payment methods and payment failures, but do "
            "not explicitly cover saved payment method management."
        )
    },
    {
        "example_id": "PT-031",
        "original_category": "PAYMENT",
        "proposed_category": "PAYMENT",
        "audit_status": "likely_model_error",
        "reason": (
            "The request describes a card being declined during checkout. "
            "This closely matches the payment_issue training intent, "
            "which includes examples of declined cards and failed "
            "online payments. The prediction as ACCOUNT is likely "
            "a model classification error."
        )
    },
    {
        "example_id": "PT-032",
        "original_category": "PAYMENT",
        "proposed_category": None,
        "audit_status": "definition_review",
        "reason": (
            "The request asks to verify whether a transaction was "
            "successfully processed. Existing PAYMENT intents cover "
            "payment methods and payment issues, but do not explicitly "
            "represent transaction status inquiries."
        )
    }
])

annotation_audit = annotation_audit[
    ~annotation_audit["example_id"].isin(
        payment_updates["example_id"]
    )
]

annotation_audit = pd.concat(
    [annotation_audit, payment_updates],
    ignore_index=True
)

annotation_audit = (
    annotation_audit
    .sort_values("example_id")
    .reset_index(drop=True)
)

annotation_audit.to_csv(audit_path, index=False)

print("PAYMENT audit records updated successfully.")

print("\nAudit status summary:")
display(annotation_audit["audit_status"].value_counts())

print("\nUpdated PAYMENT cases:")
display(
    annotation_audit[
        annotation_audit["example_id"].isin(
            ["PT-030", "PT-031", "PT-032"]
        )
    ]
)

PAYMENT audit records updated successfully.

Audit status summary:


audit_status
definition_review     9
likely_model_error    6
proposed_relabel      2
Name: count, dtype: int64


Updated PAYMENT cases:


,example_id,original_category,proposed_category,audit_status,reason
10,PT-030,PAYMENT,NaN,definition_review,The request concerns replacing a stored paymen...
11,PT-031,PAYMENT,PAYMENT,likely_model_error,The request describes a card being declined du...
12,PT-032,PAYMENT,NaN,definition_review,The request asks to verify whether a transacti...


In [50]:
# Validate annotation audit completeness

print("Total audited cases:", len(annotation_audit))

print(
    "Unique example IDs:",
    annotation_audit["example_id"].nunique()
)

print("\nAudit status distribution:")
display(
    annotation_audit["audit_status"].value_counts()
)

print("\nDuplicate example IDs:")
display(
    annotation_audit[
        annotation_audit["example_id"].duplicated(keep=False)
    ]
)

print("\nProposed category distribution:")
display(
    annotation_audit["proposed_category"].value_counts(
        dropna=False
    )
)

assert len(annotation_audit) == 17
assert annotation_audit["example_id"].nunique() == 17
assert not annotation_audit["example_id"].duplicated().any()

print("\nAudit validation passed successfully.")

Total audited cases: 17
Unique example IDs: 17

Audit status distribution:


audit_status
definition_review     9
likely_model_error    6
proposed_relabel      2
Name: count, dtype: int64


Duplicate example IDs:


,example_id,original_category,proposed_category,audit_status,reason



Proposed category distribution:


proposed_category
NaN         9
ORDER       2
SHIPPING    2
CONTACT     1
DELIVERY    1
FEEDBACK    1
PAYMENT     1
Name: count, dtype: int64


Audit validation passed successfully.


In [51]:
# Inspect audit decisions by status

print("Audit status vs proposed category:")

display(
    pd.crosstab(
        annotation_audit["audit_status"],
        annotation_audit["proposed_category"],
        dropna=False
    )
)

print("\nComplete annotation audit:")

display(
    annotation_audit[
        [
            "example_id",
            "original_category",
            "proposed_category",
            "audit_status",
            "reason"
        ]
    ].sort_values("example_id")
)

Audit status vs proposed category:


proposed_category,CONTACT,DELIVERY,FEEDBACK,ORDER,PAYMENT,SHIPPING,NaN
audit_status,,,,,,,
definition_review,0,0,0,0,0,0,9
likely_model_error,1,1,1,0,1,2,0
proposed_relabel,0,0,0,2,0,0,0



Complete annotation audit:


,example_id,original_category,proposed_category,audit_status,reason
0,PT-005,CANCEL,NaN,definition_review,The request concerns terminating an active ser...
1,PT-006,CANCEL,NaN,definition_review,The request expresses a desire to discontinue ...
2,PT-007,CANCEL,NaN,definition_review,The request concerns stopping a service before...
3,PT-008,CANCEL,ORDER,proposed_relabel,The request explicitly asks to cancel a purcha...
4,PT-011,CONTACT,CONTACT,likely_model_error,The request asks for human assistance after au...
5,PT-013,DELIVERY,DELIVERY,likely_model_error,The request asks for the estimated package arr...
6,PT-015,DELIVERY,ORDER,proposed_relabel,The request concerns an unchanged tracking pag...
7,PT-017,FEEDBACK,NaN,definition_review,Sharing an experience with customer service ma...
8,PT-019,FEEDBACK,FEEDBACK,likely_model_error,"The request suggests an improvement, which ali..."
9,PT-027,ORDER,NaN,definition_review,Missing order confirmation after checkout may ...


In [52]:
 # Inspect benchmark directory

print("Benchmark directory:", benchmark_dir)

print("\nAvailable files:")

for file in sorted(benchmark_dir.iterdir()):
    if file.is_file():
        print(file.name)

Benchmark directory: /Users/raghad/Desktop/Lumora/data/benchmark

Available files:
ambiguous_probabilities_v1.csv
ambiguous_results_v1.csv
ambiguous_summary_v1.csv
annotation_audit_v1.csv
challenge.csv
classification_errors_v1.csv
classification_metrics_v1.csv
classification_overall_metrics_v1.csv
classification_reliability_v1.csv
classification_results_v1.csv
development.csv
development_v2.csv
error_patterns_v1.csv
locked_test.csv
predeployment_test_v1.csv
score_analysis_v1.csv
score_summary_v1.csv


In [53]:
# Inspect benchmark evaluation files

files_to_inspect = [
    "predeployment_test_v1.csv",
    "classification_results_v1.csv",
    "classification_metrics_v1.csv",
    "annotation_audit_v1.csv"
]

for filename in files_to_inspect:

    file_path = benchmark_dir / filename
    df = pd.read_csv(file_path)

    print("\n" + "=" * 75)
    print(f"FILE: {filename}")
    print("=" * 75)

    print("Shape:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData types:")
    display(df.dtypes.to_frame("Data Type"))

    print("\nFirst 3 rows:")
    display(df.head(3))


FILE: predeployment_test_v1.csv
Shape: (65, 10)

Columns:
['example_id', 'text', 'test_group', 'test_type', 'expected_category', 'expected_categories', 'is_in_scope', 'requires_clarification', 'out_of_scope_type', 'annotation_rationale']

Data types:


,Data Type
example_id,str
text,str
test_group,str
test_type,str
expected_category,str
expected_categories,str
is_in_scope,bool
requires_clarification,object
out_of_scope_type,str
annotation_rationale,str



First 3 rows:


,example_id,text,test_group,test_type,expected_category,expected_categories,is_in_scope,requires_clarification,out_of_scope_type,annotation_rationale
0,PT-001,I need to update the email linked to my custom...,classification,natural,ACCOUNT,NaN,True,False,NaN,NaN
1,PT-002,Where can I change the personal details saved ...,classification,paraphrase,ACCOUNT,NaN,True,False,NaN,NaN
2,PT-003,"My profile information is outdated, and I want...",classification,contextual,ACCOUNT,NaN,True,False,NaN,NaN



FILE: classification_results_v1.csv
Shape: (44, 10)

Columns:
['example_id', 'text', 'expected_category', 'test_type', 'predicted_category', 'correct', 'top_score', 'second_prediction', 'second_score', 'prediction_margin']

Data types:


,Data Type
example_id,str
text,str
expected_category,str
test_type,str
predicted_category,str
correct,bool
top_score,float64
second_prediction,str
second_score,float64
prediction_margin,float64


... (6 outputs left out)


FILE: annotation_audit_v1.csv
Shape: (17, 5)

Columns:
['example_id', 'original_category', 'proposed_category', 'audit_status', 'reason']

Data types:


,Data Type
example_id,str
original_category,str
proposed_category,str
audit_status,str
reason,str



First 3 rows:


,example_id,original_category,proposed_category,audit_status,reason
0,PT-005,CANCEL,NaN,definition_review,The request concerns terminating an active ser...
1,PT-006,CANCEL,NaN,definition_review,The request expresses a desire to discontinue ...
2,PT-007,CANCEL,NaN,definition_review,The request concerns stopping a service before...


In [54]:
# Merge annotation audit with benchmark labels and predictions

predeployment_test = pd.read_csv(
    benchmark_dir / "predeployment_test_v1.csv"
)

classification_results = pd.read_csv(
    benchmark_dir / "classification_results_v1.csv"
)

annotation_audit = pd.read_csv(
    benchmark_dir / "annotation_audit_v1.csv"
)

# Select relevant columns
test_labels = predeployment_test[
    ["example_id", "expected_category"]
].copy()

model_predictions = classification_results[
    [
        "example_id",
        "predicted_category",
        "correct",
        "top_score",
        "prediction_margin"
    ]
].copy()

# Merge audit records with original labels
audit_analysis = annotation_audit.merge(
    test_labels,
    on="example_id",
    how="left",
    validate="one_to_one"
)

# Merge model predictions
audit_analysis = audit_analysis.merge(
    model_predictions,
    on="example_id",
    how="left",
    validate="one_to_one"
)

# Validate original labels
audit_analysis["original_label_matches"] = (
    audit_analysis["original_category"]
    == audit_analysis["expected_category"]
)

print("Audit analysis shape:", audit_analysis.shape)

print("\nOriginal label consistency:")
display(
    audit_analysis["original_label_matches"].value_counts(
        dropna=False
    )
)

print("\nPrediction availability:")
display(
    audit_analysis["predicted_category"]
    .notna()
    .value_counts()
)

print("\nComplete audit and prediction analysis:")

display(
    audit_analysis[
        [
            "example_id",
            "original_category",
            "expected_category",
            "proposed_category",
            "predicted_category",
            "correct",
            "audit_status",
            "top_score",
            "prediction_margin"
        ]
    ].sort_values("example_id")
)

Audit analysis shape: (17, 11)

Original label consistency:


original_label_matches
True    17
Name: count, dtype: int64


Prediction availability:


predicted_category
True    17
Name: count, dtype: int64


Complete audit and prediction analysis:


,example_id,original_category,expected_category,proposed_category,predicted_category,correct,audit_status,top_score,prediction_margin
0,PT-005,CANCEL,CANCEL,NaN,ACCOUNT,False,definition_review,0.190964,0.024726
1,PT-006,CANCEL,CANCEL,NaN,CONTACT,False,definition_review,0.292411,0.060313
2,PT-007,CANCEL,CANCEL,NaN,ACCOUNT,False,definition_review,0.189318,0.042279
3,PT-008,CANCEL,CANCEL,ORDER,ORDER,False,proposed_relabel,0.621440,0.527017
4,PT-011,CONTACT,CONTACT,CONTACT,ACCOUNT,False,likely_model_error,0.390343,0.259199
5,PT-013,DELIVERY,DELIVERY,DELIVERY,ACCOUNT,False,likely_model_error,0.265854,0.036926
6,PT-015,DELIVERY,DELIVERY,ORDER,REFUND,False,proposed_relabel,0.195909,0.009454
7,PT-017,FEEDBACK,FEEDBACK,NaN,CONTACT,False,definition_review,0.512101,0.131626
8,PT-019,FEEDBACK,FEEDBACK,FEEDBACK,ORDER,False,likely_model_error,0.535604,0.424788
9,PT-027,ORDER,ORDER,NaN,ACCOUNT,False,definition_review,0.273076,0.109832


In [55]:
import pandas as pd
from pathlib import Path

# Load the original classification errors
errors_path = benchmark_dir / "classification_errors_v1.csv"
classification_errors = pd.read_csv(errors_path)

# Select the audit information
audit_review = audit_analysis[
    [
        "example_id",
        "original_category",
        "predicted_category",
        "top_score",
        "prediction_margin",
    ]
].copy()

# Add the original text to each audit case
review_cases = audit_review.merge(
    classification_errors[["example_id", "text"]],
    on="example_id",
    how="left",
    validate="one_to_one"
)

# Add the audit decisions
review_cases = review_cases.merge(
    annotation_audit[
        [
            "example_id",
            "proposed_category",
            "audit_status",
            "reason",
        ]
    ],
    on="example_id",
    how="left",
    validate="one_to_one"
)

# Organize the columns
review_cases = review_cases[
    [
        "example_id",
        "text",
        "original_category",
        "predicted_category",
        "proposed_category",
        "audit_status",
        "reason",
        "top_score",
        "prediction_margin",
    ]
]

# Display all 17 cases
pd.set_option("display.max_colwidth", None)
display(review_cases)

,example_id,text,original_category,predicted_category,proposed_category,audit_status,reason,top_score,prediction_margin
0,PT-005,Please terminate my current service before the next billing cycle.,CANCEL,ACCOUNT,NaN,definition_review,"The request concerns terminating an active service before the next billing cycle. The CANCEL training intent covers cancellation fees, while ACCOUNT/delete_account covers account removal. No explicit service-termination intent was identified.",0.190964,0.024726
1,PT-006,I no longer wish to continue using this service.,CANCEL,CONTACT,NaN,definition_review,The request expresses a desire to discontinue a service without explicitly mentioning account deletion or cancellation fees. The current taxonomy does not clearly define where general service discontinuation belongs.,0.292411,0.060313
2,PT-007,"My renewal is approaching, so I want to stop the service beforehand.",CANCEL,ACCOUNT,NaN,definition_review,"The request concerns stopping a service before renewal. This may represent subscription or service cancellation, but the current taxonomy has no explicit intent for preventing service renewal.",0.189318,0.042279
3,PT-008,I would like to cancel the order I placed earlier today.,CANCEL,ORDER,ORDER,proposed_relabel,"The request explicitly asks to cancel a purchase order, matching ORDER -> cancel_order.",0.621440,0.527017
4,PT-011,"The automated replies have not resolved my issue, and I need human assistance.",CONTACT,ACCOUNT,CONTACT,likely_model_error,"The request asks for human assistance after automated replies failed, which aligns with human-agent contact.",0.390343,0.259199
5,PT-013,Can you tell me the estimated arrival date of my package?,DELIVERY,ACCOUNT,DELIVERY,likely_model_error,"The request asks for the estimated package arrival date, matching the delivery_period intent.",0.265854,0.036926
6,PT-015,"The tracking page has not changed since yesterday, and I am waiting for the parcel.",DELIVERY,REFUND,ORDER,proposed_relabel,"The request concerns an unchanged tracking page, matching ORDER -> track_order more closely.",0.195909,0.009454
7,PT-017,I would like to share my experience with your customer service.,FEEDBACK,CONTACT,NaN,definition_review,Sharing an experience with customer service may overlap with CONTACT and FEEDBACK.,0.512101,0.131626
8,PT-019,"After completing my purchase, I noticed something I would like to suggest improving.",FEEDBACK,ORDER,FEEDBACK,likely_model_error,"The request suggests an improvement, which aligns with feedback despite mentioning a purchase.",0.535604,0.424788
9,PT-027,"I completed checkout, but no confirmation appeared on the screen.",ORDER,ACCOUNT,NaN,definition_review,Missing order confirmation after checkout may relate to order status or a system/account issue.,0.273076,0.109832


In [56]:
import pandas as pd

# Define the audit decisions
audit_decisions = {
    "PT-005": "definition_review",
    "PT-006": "definition_review",
    "PT-007": "definition_review",
    "PT-008": "relabel",
    "PT-011": "model_error",
    "PT-013": "model_error",
    "PT-015": "relabel",
    "PT-017": "definition_review",
    "PT-019": "model_error",
    "PT-027": "definition_review",
    "PT-030": "definition_review",
    "PT-031": "model_error",
    "PT-032": "definition_review",
    "PT-037": "model_error",
    "PT-039": "model_error",
    "PT-041": "definition_review",
    "PT-043": "definition_review",
}

# Create a separate audit decisions table
audit_decisions_df = review_cases[
    [
        "example_id",
        "original_category",
        "predicted_category",
        "proposed_category",
        "reason",
    ]
].copy()

# Assign final audit decisions
audit_decisions_df["audit_decision"] = (
    audit_decisions_df["example_id"].map(audit_decisions)
)

# Determine the audited category
audit_decisions_df["audited_category"] = (
    audit_decisions_df["original_category"]
)

# Keep unresolved categories empty
definition_mask = (
    audit_decisions_df["audit_decision"] == "definition_review"
)

audit_decisions_df.loc[
    definition_mask, "audited_category"
] = pd.NA

# Apply only the two explicitly proposed relabels
relabel_mask = (
    audit_decisions_df["audit_decision"] == "relabel"
)

audit_decisions_df.loc[
    relabel_mask, "audited_category"
] = audit_decisions_df.loc[
    relabel_mask, "proposed_category"
]

# Rename reason column
audit_decisions_df = audit_decisions_df.rename(
    columns={"reason": "audit_reason"}
)

# Validate the audit table
assert len(audit_decisions_df) == 17
assert audit_decisions_df["example_id"].is_unique
assert audit_decisions_df["audit_decision"].notna().all()

# Save without modifying the original dataset
output_path = (
    benchmark_dir / "classification_audit_decisions_v1.csv"
)

audit_decisions_df.to_csv(output_path, index=False)

print("Audit decisions saved successfully!")
print(f"File: {output_path}")

print("\nDecision summary:")
display(
    audit_decisions_df["audit_decision"]
    .value_counts()
    .rename_axis("decision")
    .to_frame("count")
)

print("\nFinal audit decisions:")
display(audit_decisions_df)

Audit decisions saved successfully!
File: /Users/raghad/Desktop/Lumora/data/benchmark/classification_audit_decisions_v1.csv

Decision summary:


,count
decision,
definition_review,9
model_error,6
relabel,2



Final audit decisions:


,example_id,original_category,predicted_category,proposed_category,audit_reason,audit_decision,audited_category
0,PT-005,CANCEL,ACCOUNT,NaN,"The request concerns terminating an active service before the next billing cycle. The CANCEL training intent covers cancellation fees, while ACCOUNT/delete_account covers account removal. No explicit service-termination intent was identified.",definition_review,NaN
1,PT-006,CANCEL,CONTACT,NaN,The request expresses a desire to discontinue a service without explicitly mentioning account deletion or cancellation fees. The current taxonomy does not clearly define where general service discontinuation belongs.,definition_review,NaN
2,PT-007,CANCEL,ACCOUNT,NaN,"The request concerns stopping a service before renewal. This may represent subscription or service cancellation, but the current taxonomy has no explicit intent for preventing service renewal.",definition_review,NaN
3,PT-008,CANCEL,ORDER,ORDER,"The request explicitly asks to cancel a purchase order, matching ORDER -> cancel_order.",relabel,ORDER
4,PT-011,CONTACT,ACCOUNT,CONTACT,"The request asks for human assistance after automated replies failed, which aligns with human-agent contact.",model_error,CONTACT
5,PT-013,DELIVERY,ACCOUNT,DELIVERY,"The request asks for the estimated package arrival date, matching the delivery_period intent.",model_error,DELIVERY
6,PT-015,DELIVERY,REFUND,ORDER,"The request concerns an unchanged tracking page, matching ORDER -> track_order more closely.",relabel,ORDER
7,PT-017,FEEDBACK,CONTACT,NaN,Sharing an experience with customer service may overlap with CONTACT and FEEDBACK.,definition_review,NaN
8,PT-019,FEEDBACK,ORDER,FEEDBACK,"The request suggests an improvement, which aligns with feedback despite mentioning a purchase.",model_error,FEEDBACK
9,PT-027,ORDER,ACCOUNT,NaN,Missing order confirmation after checkout may relate to order status or a system/account issue.,definition_review,NaN


In [57]:
import pandas as pd
from pathlib import Path

# Load the original dataset
project_root = Path.home() / "Desktop" / "Lumora"

data_path = (
    project_root
    / "data"
    / "raw"
    / "customer_support_tickets.csv"
)

df_original = pd.read_csv(data_path)

# Categories requiring taxonomy review
categories_to_review = [
    "CANCEL",
    "CONTACT",
    "FEEDBACK",
    "ORDER",
    "PAYMENT",
    "DELIVERY",
    "SHIPPING",
    "SUBSCRIPTION",
]

taxonomy_data = df_original[
    df_original["category"].isin(categories_to_review)
].copy()

# Count intents within each category
intent_summary = (
    taxonomy_data
    .groupby(["category", "intent"])
    .size()
    .reset_index(name="sample_count")
    .sort_values(
        ["category", "sample_count"],
        ascending=[True, False]
    )
)

print("Taxonomy Intent Summary:")
display(intent_summary)

# Show actual training examples for each intent
print("\nOriginal Training Examples:")

for category in categories_to_review:
    print(f"\n{'=' * 60}")
    print(f"CATEGORY: {category}")
    print(f"{'=' * 60}")

    category_data = taxonomy_data[
        taxonomy_data["category"] == category
    ]

    for intent in category_data["intent"].unique():
        examples = category_data[
            category_data["intent"] == intent
        ]["instruction"].head(3)

        print(f"\nIntent: {intent}")

        for i, example in enumerate(examples, 1):
            print(f"{i}. {example}")

Taxonomy Intent Summary:


,category,intent,sample_count
0,CANCEL,check_cancellation_fee,950
1,CONTACT,contact_customer_service,1000
2,CONTACT,contact_human_agent,999
4,DELIVERY,delivery_period,999
3,DELIVERY,delivery_options,995
5,FEEDBACK,complaint,1000
6,FEEDBACK,review,997
7,ORDER,cancel_order,998
9,ORDER,place_order,998
8,ORDER,change_order,997



Original Training Examples:

CATEGORY: CANCEL

Intent: check_cancellation_fee
1. I can't ifnd the bloody termination charge, I need help
2. what is the fee for canceling the contract?
3. could I check the withdrawal charge?

CATEGORY: CONTACT

Intent: contact_customer_service
1. I want help to speak to customer support
2. do you have a free number to talk to customer assistance?
3. I try to see what hours customer service aailable is

Intent: contact_human_agent
1. I do not understand you, help me chatting with someone
2. can I talk to a person?
3. I can't speak with a live agent

CATEGORY: FEEDBACK

Intent: complaint
1. help me to file a claim
2. where can i make a consumer claim against ur organization
3. want help to make a consumer complaint against ur company

Intent: review
1. uhave a method to leave an opinion about ur services
2. I need help to leave a review about your products
3. do you have an address to leave some feedback?

CATEGORY: ORDER

Intent: cancel_order
1. questio

In [58]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

# Prepare category documents
category_texts = (
    df_original
    .groupby("category")["instruction"]
    .apply(lambda x: " ".join(x.astype(str)))
)

# Convert category texts into TF-IDF vectors
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    stop_words="english",
    min_df=2
)

category_vectors = vectorizer.fit_transform(category_texts)

# Calculate pairwise cosine similarity
similarity_matrix = cosine_similarity(category_vectors)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=category_texts.index,
    columns=category_texts.index
)

# Display category similarity
display(similarity_df.round(3))

category,ACCOUNT,CANCEL,CONTACT,DELIVERY,FEEDBACK,INVOICE,ORDER,PAYMENT,REFUND,SHIPPING,SUBSCRIPTION
category,,,,,,,,,,,
ACCOUNT,1.000,0.242,0.217,0.147,0.247,0.202,0.103,0.517,0.395,0.150,0.394
CANCEL,0.242,1.000,0.144,0.170,0.126,0.132,0.051,0.187,0.362,0.037,0.135
CONTACT,0.217,0.144,1.000,0.105,0.440,0.237,0.063,0.171,0.255,0.052,0.142
DELIVERY,0.147,0.170,0.105,1.000,0.111,0.088,0.226,0.329,0.395,0.286,0.103
FEEDBACK,0.247,0.126,0.440,0.111,1.000,0.130,0.072,0.208,0.288,0.099,0.586
INVOICE,0.202,0.132,0.237,0.088,0.130,1.000,0.046,0.132,0.226,0.036,0.120
ORDER,0.103,0.051,0.063,0.226,0.072,0.046,1.000,0.060,0.170,0.019,0.091
PAYMENT,0.517,0.187,0.171,0.329,0.208,0.132,0.060,1.000,0.337,0.071,0.200
REFUND,0.395,0.362,0.255,0.395,0.288,0.226,0.170,0.337,1.000,0.084,0.363


In [59]:
# Exact duplicate analysis

duplicate_analysis = (
    df_original
    .groupby("instruction")
    .agg(
        occurrences=("category", "size"),
        unique_categories=("category", "nunique"),
        unique_intents=("intent", "nunique")
    )
    .reset_index()
)

# Separate repeated instructions
repeated_instructions = duplicate_analysis[
    duplicate_analysis["occurrences"] > 1
].copy()

# Check conflicting labels
conflicting_labels = repeated_instructions[
    (repeated_instructions["unique_categories"] > 1) |
    (repeated_instructions["unique_intents"] > 1)
].copy()

print("Total records:", len(df_original))

print(
    "Unique instructions:",
    df_original["instruction"].nunique()
)

print(
    "Repeated instruction groups:",
    len(repeated_instructions)
)

print(
    "Conflicting instruction groups:",
    len(conflicting_labels)
)

print("\nRepeated instructions:")
display(
    repeated_instructions
    .sort_values("occurrences", ascending=False)
    .head(20)
)

print("\nConflicting instructions:")
display(conflicting_labels.head(20))

Total records: 26872
Unique instructions: 24635
Repeated instruction groups: 989
Conflicting instruction groups: 0

Repeated instructions:


,instruction,occurrences,unique_categories,unique_intents
8860,can I place an order from {{Delivery City}}?,8,1,1
11539,do you shp to {{Delivery City}}?,8,1,1
9091,can i place an order from {{Delivery City}},8,1,1
9082,can i order from {{Delivery City}},8,1,1
11466,do you deliver to {{Delivery City}}?,8,1,1
11465,do you deliver to {{Delivery City}} ?,8,1,1
8855,can I order from {{Delivery City}}?,8,1,1
18840,is it possibleto place an order from {{Delivery City}},8,1,1
18837,is it possible toorder from {{Delivery City}},8,1,1
20828,shiping to {{Delivery City}},8,1,1



Conflicting instructions:


,instruction,occurrences,unique_categories,unique_intents


In [60]:
import re

def normalize_instruction(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text


df_original["normalized_instruction"] = (
    df_original["instruction"]
    .apply(normalize_instruction)
)

normalized_analysis = (
    df_original
    .groupby("normalized_instruction")
    .agg(
        occurrences=("category", "size"),
        unique_categories=("category", "nunique"),
        unique_intents=("intent", "nunique"),
        original_variations=("instruction", "nunique")
    )
    .reset_index()
)

normalized_duplicates = normalized_analysis[
    normalized_analysis["occurrences"] > 1
].copy()

normalized_conflicts = normalized_duplicates[
    (normalized_duplicates["unique_categories"] > 1) |
    (normalized_duplicates["unique_intents"] > 1)
].copy()

print("Unique normalized instructions:", len(normalized_analysis))
print("Repeated normalized groups:", len(normalized_duplicates))
print("Normalized conflicting groups:", len(normalized_conflicts))

print("\nGroups with multiple original variations:")
display(
    normalized_duplicates[
        normalized_duplicates["original_variations"] > 1
    ]
    .sort_values("occurrences", ascending=False)
    .head(20)
)

print("\nNormalized conflicts:")
display(normalized_conflicts.head(20))

Unique normalized instructions: 24010
Repeated normalized groups: 1371
Normalized conflicting groups: 0

Groups with multiple original variations:


,normalized_instruction,occurrences,unique_categories,unique_intents,original_variations
18143,is it possible to order from delivery city,16,1,1,2
18155,is it possible to place an order from delivery city,16,1,1,2
897,can i place an order from delivery city,16,1,1,2
886,can i order from delivery city,16,1,1,2
18157,is it possible to place an order from delivery country,12,1,1,2
18145,is it possible to order from delivery country,12,1,1,2
3462,do you ship to delivery country,12,1,1,2
898,can i place an order from delivery country,12,1,1,2
887,can i order from delivery country,12,1,1,2
10651,i expect a compensation of currency symbolrefund amount,10,1,1,2



Normalized conflicts:


,normalized_instruction,occurrences,unique_categories,unique_intents,original_variations


In [61]:
# Normalized duplicate impact analysis

total_records = len(df_original)

repeated_records = normalized_duplicates[
    normalized_duplicates["occurrences"] > 1
]["occurrences"].sum()

excess_records = (
    normalized_duplicates["occurrences"] - 1
).sum()

records_in_repeated_groups = repeated_records

print(f"Total records: {total_records:,}")

print(
    f"Records in repeated normalized groups: "
    f"{records_in_repeated_groups:,}"
)

print(
    f"Excess repeated records: {excess_records:,}"
)

print(
    f"Percentage of records in repeated groups: "
    f"{records_in_repeated_groups / total_records * 100:.2f}%"
)

print(
    f"Percentage of excess repeated records: "
    f"{excess_records / total_records * 100:.2f}%"
)

print(
    f"Records in unique normalized groups: "
    f"{total_records - records_in_repeated_groups:,}"
)

Total records: 26,872
Records in repeated normalized groups: 4,233
Excess repeated records: 2,862
Percentage of records in repeated groups: 15.75%
Percentage of excess repeated records: 10.65%
Records in unique normalized groups: 22,639


In [62]:
from sklearn.model_selection import GroupShuffleSplit

# Prepare features, labels, and groups

X = df_original["instruction"].copy()
y = df_original["category"].copy()

groups = df_original["instruction"].apply(normalize_instruction)

# First split: 70% training, 30% temporary

first_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    first_split.split(X, y, groups)
)

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]
groups_train = groups.iloc[train_idx]

X_temp = X.iloc[temp_idx]
y_temp = y.iloc[temp_idx]
groups_temp = groups.iloc[temp_idx]

# Second split: 15% validation, 15% test

second_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_relative_idx, test_relative_idx = next(
    second_split.split(X_temp, y_temp, groups_temp)
)

X_val = X_temp.iloc[val_relative_idx]
y_val = y_temp.iloc[val_relative_idx]
groups_val = groups_temp.iloc[val_relative_idx]

X_test = X_temp.iloc[test_relative_idx]
y_test = y_temp.iloc[test_relative_idx]
groups_test = groups_temp.iloc[test_relative_idx]

# Display split sizes

split_summary = pd.DataFrame({
    "Dataset": ["Training", "Validation", "Test"],
    "Records": [len(X_train), len(X_val), len(X_test)],
    "Percentage": [
        len(X_train) / len(X) * 100,
        len(X_val) / len(X) * 100,
        len(X_test) / len(X) * 100
    ],
    "Unique Groups": [
        groups_train.nunique(),
        groups_val.nunique(),
        groups_test.nunique()
    ]
})

display(split_summary)

,Dataset,Records,Percentage,Unique Groups
0,Training,18798,69.953855,16807
1,Validation,4041,15.037958,3601
2,Test,4033,15.008187,3602


In [63]:
# 1. Verify group separation

train_groups = set(groups_train)
val_groups = set(groups_val)
test_groups = set(groups_test)

print("Training ∩ Validation:", len(train_groups & val_groups))
print("Training ∩ Test:", len(train_groups & test_groups))
print("Validation ∩ Test:", len(val_groups & test_groups))


# 2. Combine labels for distribution analysis

split_labels = pd.DataFrame({
    "Training": y_train.value_counts(),
    "Validation": y_val.value_counts(),
    "Test": y_test.value_counts()
}).fillna(0).astype(int)

# Sort categories alphabetically
split_labels = split_labels.sort_index()

print("\nCategory counts:")
display(split_labels)


# 3. Calculate percentage distribution within each split

split_percentages = (
    split_labels
    .div(split_labels.sum(axis=0), axis=1)
    * 100
)

print("\nCategory percentages:")
display(split_percentages.round(2))


# 4. Check whether all categories exist in every split

print("\nNumber of categories in each split:")
print("Training:", y_train.nunique())
print("Validation:", y_val.nunique())
print("Test:", y_test.nunique())

Training ∩ Validation: 0
Training ∩ Test: 0
Validation ∩ Test: 0

Category counts:


,Training,Validation,Test
category,,,
ACCOUNT,4106,981,899
CANCEL,671,150,129
CONTACT,1411,295,293
DELIVERY,1416,289,289
FEEDBACK,1413,275,309
INVOICE,1425,271,303
ORDER,2795,573,620
PAYMENT,1396,312,290
REFUND,2068,449,475



Category percentages:


,Training,Validation,Test
category,,,
ACCOUNT,21.84,24.28,22.29
CANCEL,3.57,3.71,3.20
CONTACT,7.51,7.30,7.27
DELIVERY,7.53,7.15,7.17
FEEDBACK,7.52,6.81,7.66
INVOICE,7.58,6.71,7.51
ORDER,14.87,14.18,15.37
PAYMENT,7.43,7.72,7.19
REFUND,11.00,11.11,11.78



Number of categories in each split:
Training: 11
Validation: 11
Test: 11


In [64]:
# Compare split distributions against original distribution

original_distribution = (
    y.value_counts(normalize=True) * 100
).sort_index()

distribution_comparison = pd.DataFrame({
    "Original": original_distribution,
    "Training": y_train.value_counts(normalize=True) * 100,
    "Validation": y_val.value_counts(normalize=True) * 100,
    "Test": y_test.value_counts(normalize=True) * 100
}).fillna(0)

distribution_comparison["Validation Difference"] = (
    distribution_comparison["Validation"]
    - distribution_comparison["Original"]
)

distribution_comparison["Test Difference"] = (
    distribution_comparison["Test"]
    - distribution_comparison["Original"]
)

display(distribution_comparison.round(2))

,Original,Training,Validation,Test,Validation Difference,Test Difference
category,,,,,,
ACCOUNT,22.28,21.84,24.28,22.29,2.00,0.02
CANCEL,3.54,3.57,3.71,3.20,0.18,-0.34
CONTACT,7.44,7.51,7.30,7.27,-0.14,-0.17
DELIVERY,7.42,7.53,7.15,7.17,-0.27,-0.25
FEEDBACK,7.43,7.52,6.81,7.66,-0.63,0.23
INVOICE,7.44,7.58,6.71,7.51,-0.73,0.07
ORDER,14.84,14.87,14.18,15.37,-0.66,0.53
PAYMENT,7.44,7.43,7.72,7.19,0.29,-0.24
REFUND,11.13,11.00,11.11,11.78,-0.02,0.64


In [65]:
# Save reproducible split indices

import json
from pathlib import Path

split_indices = {
    "train": train_idx.tolist(),
    "validation": temp_idx[val_relative_idx].tolist(),
    "test": temp_idx[test_relative_idx].tolist()
}

split_path = Path("../data/processed")
split_path.mkdir(parents=True, exist_ok=True)

with open(
    split_path / "lumora_split_indices_v1.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(split_indices, file, indent=2)

print("Split indices saved successfully.")
print("File:", split_path / "lumora_split_indices_v1.json")

Split indices saved successfully.
File: ../data/processed/lumora_split_indices_v1.json
